In [ ]:
import gc
import json
import math
import os
import time
import torch
import csv
import random
import hashlib

from __future__ import annotations
from pathlib import Path
from typing import Optional, Any
from dataclasses import asdict, dataclass, fields, is_dataclass
from datetime import datetime
from types import SimpleNamespace
from torch.utils.data import DataLoader, Dataset


import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

In [1]:
## This part is a full copy of models configs required to run the experiments


@dataclass
class ToyTransformerConfig:

    d_model: int = 384
    n_heads: int = 6
    d_head: int = 64
    n_layers: int = 6
    ffn_mult: float = 4.0

    vocab_size: int = 50257
    max_seq_len: int = 1024

    dropout: float = 0.0
    tie_embeddings: bool = True
    norm_eps: float = 1e-6

    @classmethod
    def full(cls, **overrides) -> "ToyTransformerConfig":
        # Stronger baseline. Not exactly 54M because GPT-2 vocab embedding is large.
        defaults = dict(
            d_model=512,
            n_heads=8,
            d_head=64,
            n_layers=8,
            ffn_mult=4.0,
        )
        defaults.update(overrides)
        return cls(**defaults)

    @classmethod
    def half(cls, **overrides) -> "ToyTransformerConfig":
        # Half baseline.
        defaults = dict(
            d_model=384,
            n_heads=6,
            d_head=64,
            n_layers=6,
            ffn_mult=4.0,
        )
        defaults.update(overrides)
        return cls(**defaults)

    @classmethod
    def half_param_matched(cls, **overrides) -> "ToyTransformerConfig":
        # More parameter-conscious comparison against ToyMamba2/ToyKimi half runs.
        defaults = dict(
            d_model=384,
            n_heads=6,
            d_head=64,
            n_layers=4,
            ffn_mult=4.0,
        )
        defaults.update(overrides)
        return cls(**defaults)

    @classmethod
    def quarter(cls, **overrides) -> "ToyTransformerConfig":
        defaults = dict(
            d_model=256,
            n_heads=4,
            d_head=64,
            n_layers=4,
            ffn_mult=4.0,
        )
        defaults.update(overrides)
        return cls(**defaults)

    def validate(self) -> None:
        assert self.d_model == self.n_heads * self.d_head, (
            f"d_model must equal n_heads * d_head, got "
            f"{self.d_model} != {self.n_heads} * {self.d_head}"
        )
        assert self.max_seq_len > 0
        assert self.vocab_size > 0
        assert self.n_layers > 0
        assert self.ffn_mult > 0

    @property
    def ffn_dim(self) -> int:
        return int(self.d_model * self.ffn_mult)

@dataclass
class ToyMamba2Config:
    # Mini Mamba-2-style model config.

    # This model uses:
      # SSD block decomposition
      # scalar-identity A per head
      # MVA/GQA-style sharing of B/C through n_groups
      # parallel input projection producing z, xBC, dt
      # causal depthwise conv over [x, B, C]
      # D skip connection
      # gate before normalization in the Mamba block

    # Model dims
    d_model: int = 512
    d_head: int = 64
    d_state: int = 64
    expand: int = 2

    # Architecture
    n_layers: int = 17
    n_groups: int = 1
    conv_dim: int = 4
    block_len: int = 64

    # Vocab / seq
    vocab_size: int = 50257
    max_seq_len: int = 1024

    # Regularization
    dropout: float = 0.0

    # Misc
    tie_embeddings: bool = True
    norm_eps: float = 1e-6

    @classmethod
    def full(cls, **overrides) -> "ToyMamba2Config":
        defaults = dict(d_model=512, n_layers=17, d_head=64, d_state=64, expand=2)
        defaults.update(overrides)
        return cls(**defaults)

    @classmethod
    def half(cls, **overrides) -> "ToyMamba2Config":
        defaults = dict(d_model=384, n_layers=8, d_head=64, d_state=64, expand=2)
        defaults.update(overrides)
        return cls(**defaults)

    @classmethod
    def quarter(cls, **overrides) -> "ToyMamba2Config":
        defaults = dict(d_model=256, n_layers=4, d_head=64, d_state=64, expand=2)
        defaults.update(overrides)
        return cls(**defaults)

    @property
    def d_inner(self) -> int:
        return self.d_model * self.expand

    @property
    def n_heads(self) -> int:
        return self.d_inner // self.d_head

    def validate(self) -> None:
        assert self.d_model > 0
        assert self.d_head > 0
        assert self.d_state > 0
        assert self.expand > 0
        assert self.n_layers > 0
        assert self.n_groups > 0
        assert self.conv_dim >= 1
        assert self.block_len >= 1
        assert self.max_seq_len >= 1
        assert self.d_inner % self.d_head == 0, "d_inner must be divisible by d_head"
        assert self.n_heads % self.n_groups == 0, "n_heads must be divisible by n_groups"

@dataclass
class ToyKimiConfig:

    #Preserved paper-inspired components:
      # KDA with channel-wise decay alpha
      # 3:1 KDA:global-attention ratio
      # ShortConv before q/k/v
      # L2-normalized q/k
      # sigmoid output gate
      # NoPE full attention layers
      # dense SwiGLU FFN replacing MoE at this scale

    # Model dims
    d_model: int = 512
    n_heads: int = 8
    d_k: int = 64
    d_v: int = 64

    # Architecture
    n_blocks: int = 2
    kda_per_block: int = 3
    ffn_mult: float = 3.0

    # Sequence / chunking
    vocab_size: int = 50257
    max_seq_len: int = 1024
    chunk_size: int = 64

    # KDA-specific
    conv_kernel: int = 4
    gate_rank: int = 64

    # Regularization
    dropout: float = 0.0

    # Misc
    tie_embeddings: bool = True
    norm_eps: float = 1e-6

    @classmethod
    def full(cls, **overrides) -> "ToyKimiConfig":
        defaults = dict(
            d_model=512,
            n_heads=8,
            d_k=64,
            d_v=64,
            n_blocks=2,
            ffn_mult=3.0,
            gate_rank=64,
        )
        defaults.update(overrides)
        return cls(**defaults)

    @classmethod
    def half(cls, **overrides) -> "ToyKimiConfig":
        defaults = dict(
            d_model=384,
            n_heads=6,
            d_k=64,
            d_v=64,
            n_blocks=1,
            ffn_mult=3.0,
            gate_rank=64,
        )
        defaults.update(overrides)
        return cls(**defaults)

    @classmethod
    def quarter(cls, **overrides) -> "ToyKimiConfig":
        defaults = dict(
            d_model=256,
            n_heads=4,
            d_k=64,
            d_v=64,
            n_blocks=1,
            ffn_mult=2.5,
            gate_rank=64,
        )
        defaults.update(overrides)
        return cls(**defaults)

    @property
    def ffn_dim(self) -> int:
        return int(self.d_model * self.ffn_mult)

    @property
    def n_attn_layers(self) -> int:
        return self.n_blocks * (self.kda_per_block + 1)

    @property
    def n_kda_layers(self) -> int:
        return self.n_blocks * self.kda_per_block

    @property
    def n_full_attention_layers(self) -> int:
        return self.n_blocks

    def validate(self) -> None:
        assert self.d_model > 0
        assert self.n_heads > 0
        assert self.d_k > 0
        assert self.d_v > 0
        assert self.n_blocks > 0
        assert self.kda_per_block > 0
        assert self.ffn_dim > 0
        assert self.chunk_size > 0
        assert self.conv_kernel >= 1
        assert self.gate_rank > 0
        assert self.max_seq_len > 0
        assert self.vocab_size > 0

@dataclass
class RunConfig:
    model: str = "transformer"
    model_size: str = "half"
    dataset: str = "tinystories"
    max_tokens: Optional[int] = None

    val_monitor_fraction: float = 0.02
    val_ablation_fraction: float = 0.015
    test_replication_fraction: float = 0.015
    val_monitor_max_tokens: Optional[int] = 2_000_000
    val_ablation_max_tokens: Optional[int] = 1_000_000
    test_replication_max_tokens: Optional[int] = 1_000_000

    seq_len: int = 1024
    chunk_size: int = 64
    block_len: int = 64
    batch_size: int = 24
    eval_batch_size: Optional[int] = None
    grad_accum: int = 1
    num_workers: int = 2

    lr: float = 3e-4
    weight_decay: float = 0.01
    dropout: float = 0.0
    warmup_steps: int = 1000
    max_steps: int = 100_000
    grad_clip: float = 1.0

    log_every: int = 100
    eval_every: int = 500
    eval_batches: int = 50
    final_eval_batches: int = 200
    save_every: int = 5000
    save_dir: str = "runs/checkpoints"
    log_dir: str = "runs/logs"

    early_stopping_patience: int = 8
    early_stopping_min_delta: float = 0.002
    early_stopping_warmup_evals: int = 3
    plateau_window: int = 5
    plateau_min_relative_improvement: float = 0.002

    run_correctness_tests: bool = True
    sample_generation_mode: str = "recurrent"
    no_optimized_prefill: bool = False
    test_ssd: bool = False
    test_recurrent: bool = False


def make_default_config(model: str = "transformer") -> RunConfig:
    cfg = RunConfig(model=model)
    if model == "transformer":
        cfg.model_size = "half"
        cfg.batch_size = 24
    elif model == "mamba2":
        cfg.model_size = "half"
        cfg.batch_size = 24
        cfg.test_ssd = True
        cfg.test_recurrent = True
        cfg.run_correctness_tests = False
    elif model == "kimi_linear":
        cfg.model_size = "half"
        cfg.batch_size = 24
    else:
        raise ValueError(f"Unknown model '{model}'.")
    return cfg


def namespace_to_run_config(args: object | None, model: str | None = None) -> RunConfig:
    if args is None:
        return make_default_config(model or "transformer")
    if isinstance(args, RunConfig):
        if model is not None:
            args.model = model
        return args
    cfg = make_default_config(model or getattr(args, "model", "transformer"))
    for key in cfg.__dataclass_fields__:
        if hasattr(args, key):
            setattr(cfg, key, getattr(args, key))
        upper = key.upper()
        if hasattr(args, upper):
            setattr(cfg, key, getattr(args, upper))
    if model is not None:
        cfg.model = model
    return cfg


def make_colab_config(model: str = "transformer") -> SimpleNamespace:
    cfg = make_default_config(model)
    return SimpleNamespace(**cfg.__dict__)

In [2]:
# Core KIMI modules


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dtype = x.dtype
        x_float = x.float()
        rms = torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        y = x_float * rms * self.weight.float()
        return y.to(dtype)


class ShortConv(nn.Module):
    # Causal depthwise 1D convolution used before q/k/v.

    def __init__(self, channels: int, kernel_size: int = 4):
        super().__init__()
        self.channels = channels
        self.kernel_size = kernel_size
        self.pad_len = kernel_size - 1
        self.conv = nn.Conv1d(
            channels,
            channels,
            kernel_size,
            padding=0,
            groups=channels,
            bias=False,
        )

    def allocate_state(self, batch_size: int, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
        return torch.zeros(batch_size, self.channels, self.pad_len, device=device, dtype=dtype)

    def final_state_from_full_input(self, x: torch.Tensor) -> torch.Tensor:
        #Returns the raw pre-conv tail needed for recurrent decoding.

        #x: [B, T, channels]
        #state: [B, channels, kernel_size - 1]

        B, T, C = x.shape
        if self.pad_len == 0:
            return x.new_zeros(B, C, 0)
        if T >= self.pad_len:
            return x[:, -self.pad_len :, :].transpose(1, 2).contiguous()
        pad = x.new_zeros(B, self.pad_len - T, C)
        return torch.cat([pad, x], dim=1).transpose(1, 2).contiguous()

    def forward_full(self, x: torch.Tensor) -> torch.Tensor:
        xt = x.transpose(1, 2)
        xt = F.pad(xt, (self.pad_len, 0))
        return self.conv(xt).transpose(1, 2)

    def step(self, x_t: torch.Tensor, state: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        #One-token recurrent convolution.

        #x_t: [B, 1, C]
        #state: [B, C, K-1]

        xt = x_t.transpose(1, 2)  # [B, C, 1]
        if self.pad_len > 0:
            window = torch.cat([state, xt], dim=-1)
            new_state = window[:, :, 1:].contiguous()
        else:
            window = xt
            new_state = state

        weight = self.conv.weight[:, 0, :].to(window.dtype)  # [C, K]
        out = (window * weight.unsqueeze(0)).sum(dim=-1)
        if self.conv.bias is not None:
            out = out + self.conv.bias.to(out.dtype)
        return out.unsqueeze(1), new_state

    def forward(
        self,
        x: torch.Tensor,
        state: Optional[torch.Tensor] = None,
    ) -> tuple[torch.Tensor, Optional[torch.Tensor]]:
        if state is None:
            return self.forward_full(x), None
        return self.step(x, state)


# KDA core


def _to_chunks(x: torch.Tensor, chunk_size: int) -> torch.Tensor:
    B, T, H, D = x.shape
    N = T // chunk_size
    return x.reshape(B, N, chunk_size, H, D).permute(0, 3, 1, 2, 4).contiguous()


def step_kda(
    q: torch.Tensor,          # [B, 1, H, dk]
    k: torch.Tensor,          # [B, 1, H, dk]
    v: torch.Tensor,          # [B, 1, H, dv]
    log_alpha: torch.Tensor,  # [B, 1, H, dk]
    beta: torch.Tensor,       # [B, 1, H, 1]
    state: torch.Tensor,      # [B, H, dk, dv]
    scale_q: bool = True,
) -> tuple[torch.Tensor, torch.Tensor]:
    q_t = q.squeeze(1)
    k_t = k.squeeze(1)
    v_t = v.squeeze(1)
    alpha = log_alpha.squeeze(1).exp()
    beta_t = beta.squeeze(1)

    # Diag(alpha_t) S_{t-1}
    S_pre = state.float() * alpha.float().unsqueeze(-1)

    # S_t = S_pre - beta * k * (k^T S_pre - v)^T
    kTS = torch.einsum("bhk,bhkv->bhv", k_t.float(), S_pre)
    correction = beta_t.float() * (kTS - v_t.float())
    S = S_pre - torch.einsum("bhk,bhv->bhkv", k_t.float(), correction)

    q_used = q_t.float() * (q.shape[-1] ** -0.5 if scale_q else 1.0)
    o_t = torch.einsum("bhkv,bhk->bhv", S, q_used)
    return o_t.unsqueeze(1).to(q.dtype), S.to(q.dtype)


@torch.no_grad()
def kda_recurrent_reference(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    log_alpha: torch.Tensor,
    beta: torch.Tensor,
    initial_state: Optional[torch.Tensor] = None,
    scale_q: bool = True,
) -> tuple[torch.Tensor, torch.Tensor]:
    # Slow recurrent reference for correctness tests
    B, T, H, dk = q.shape
    dv = v.shape[-1]
    if initial_state is None:
        state = torch.zeros(B, H, dk, dv, device=q.device, dtype=q.dtype)
    else:
        state = initial_state
    outs = []
    for t in range(T):
        o_t, state = step_kda(
            q[:, t : t + 1],
            k[:, t : t + 1],
            v[:, t : t + 1],
            log_alpha[:, t : t + 1],
            beta[:, t : t + 1],
            state,
            scale_q=scale_q,
        )
        outs.append(o_t)
    return torch.cat(outs, dim=1), state


def chunk_kda(
    q: torch.Tensor,          # [B, T, H, dk]
    k: torch.Tensor,          # [B, T, H, dk]
    v: torch.Tensor,          # [B, T, H, dv]
    log_alpha: torch.Tensor,  # [B, T, H, dk]
    beta: torch.Tensor,       # [B, T, H, 1]
    chunk_size: int,
    initial_state: Optional[torch.Tensor] = None,
    scale_q: bool = True,
) -> tuple[torch.Tensor, torch.Tensor]:


    # implementation is intentionally explicit and tested against the recurrent reference.
    # Accumulations are done in float32 for stability.
    orig_dtype = q.dtype
    B, T_orig, H, dk = q.shape
    dv = v.shape[-1]
    C = chunk_size

    assert k.shape == (B, T_orig, H, dk)
    assert v.shape == (B, T_orig, H, dv)
    assert log_alpha.shape == (B, T_orig, H, dk)
    assert beta.shape == (B, T_orig, H, 1)

    pad = (C - T_orig % C) % C
    if pad > 0:
        q = F.pad(q, (0, 0, 0, 0, 0, pad))
        k = F.pad(k, (0, 0, 0, 0, 0, pad))
        v = F.pad(v, (0, 0, 0, 0, 0, pad))
        log_alpha = F.pad(log_alpha, (0, 0, 0, 0, 0, pad))
        beta = F.pad(beta, (0, 0, 0, 0, 0, pad), value=0.0)

    T = T_orig + pad
    N_chunks = T // C

    q_c = _to_chunks(q, C)                                  # [B,H,N,C,dk]
    k_c = _to_chunks(k, C)
    v_c = _to_chunks(v, C)
    g = _to_chunks(log_alpha, C).float()                     # [B,H,N,C,dk]
    beta_c = _to_chunks(beta, C).float().squeeze(-1)         # [B,H,N,C]

    if scale_q:
        q_c = q_c * (dk ** -0.5)

    gc = g.cumsum(dim=3)                                     # cumulative log decay inside chunk

    mask_lower = torch.tril(torch.ones(C, C, device=q.device, dtype=torch.bool), diagonal=-1)
    mask_causal = torch.tril(torch.ones(C, C, device=q.device, dtype=torch.bool), diagonal=0)
    zero = torch.tensor(0.0, device=q.device, dtype=torch.float32)

    # KDA uses products of cumulative decays and their inverse. Clamp inverse side to avoid inf in early random training.
    g_exp = gc.exp()
    neg_g_exp = (-gc).clamp(max=80.0).exp()

    # Akk[row=i, col=j] = <k_i * gamma_i, k_j / gamma_j>, strict lower later.
    kp = k_c * g_exp.to(k_c.dtype)
    km = k_c * neg_g_exp.to(k_c.dtype)
    Akk = torch.matmul(kp, km.transpose(-1, -2)).float()
    Akk = torch.where(mask_lower, Akk, zero)

    # M = (I + StrictTril(Diag(beta) Akk))^{-1} Diag(beta).
    # Use forward-substitution on a strictly-lower triangular matrix.
    M = -(Akk * beta_c.unsqueeze(-1))
    for i in range(1, C):
        M[..., i, :i] = M[..., i, :i].clone() + (
            M[..., i, :, None].clone() * M[..., :, :i].clone()
        ).sum(dim=-2)
    eye = torch.eye(C, device=q.device, dtype=torch.float32)
    M = M + eye
    M = M * beta_c.unsqueeze(-2)

    W = torch.matmul(M.to(k_c.dtype), kp).float()            # [B,H,N,C,dk]
    U = torch.matmul(M.to(v_c.dtype), v_c).float()           # [B,H,N,C,dv]

    if initial_state is None:
        S = torch.zeros(B, H, dk, dv, device=q.device, dtype=torch.float32)
    else:
        S = initial_state.float()

    outputs = []
    qp = q_c * g_exp.to(q_c.dtype)

    for n in range(N_chunks):
        qpn = qp[:, :, n]
        kmn = km[:, :, n]
        kn = k_c[:, :, n].float()
        gn = gc[:, :, n]
        Wn = W[:, :, n]
        Un = U[:, :, n]

        Aqk = torch.matmul(qpn, kmn.transpose(-1, -2)).float()
        Aqk = torch.where(mask_causal, Aqk, zero)

        # Pseudo-value term U - W S.
        pseudo_v = Un - torch.einsum("bhck,bhkv->bhcv", Wn, S)
        inter = torch.einsum("bhck,bhkv->bhcv", qpn.float(), S)
        intra = torch.matmul(Aqk, pseudo_v)
        outputs.append(inter + intra)

        # Chunk state update.
        g_last = gn[:, :, -1:]                              # [B,H,1,dk]
        S = S * g_last.squeeze(2).exp().unsqueeze(-1)
        k_decayed = (g_last - gn).exp() * kn
        S = S + torch.einsum("bhck,bhcv->bhkv", k_decayed, pseudo_v)

    out = torch.stack(outputs, dim=2)                        # [B,H,N,C,dv]
    out = out.reshape(B, H, T, dv).permute(0, 2, 1, 3).contiguous()
    if pad > 0:
        out = out[:, :T_orig]
    return out.to(orig_dtype), S.to(orig_dtype)


# Caches

@dataclass
class KDALayerCache:
    state: torch.Tensor      # [B, H, dk, dv]
    conv_q: torch.Tensor     # [B, H*dk, K-1]
    conv_k: torch.Tensor     # [B, H*dk, K-1]
    conv_v: torch.Tensor     # [B, H*dv, K-1]


@dataclass
class MLACache:
    k: torch.Tensor          # [B, H, T, dk]
    v: torch.Tensor          # [B, H, T, dv]


@dataclass
class BlockCache:
    kda_caches: list[KDALayerCache]
    mla_cache: Optional[MLACache]


# Layers

class KDALayer(nn.Module):
    def __init__(self, cfg: ToyKimiConfig):
        super().__init__()
        cfg.validate()
        d, h, dk, dv = cfg.d_model, cfg.n_heads, cfg.d_k, cfg.d_v
        self.cfg = cfg
        self.h = h
        self.dk = dk
        self.dv = dv

        self.W_q = nn.Linear(d, h * dk, bias=False)
        self.W_k = nn.Linear(d, h * dk, bias=False)
        self.W_v = nn.Linear(d, h * dv, bias=False)

        self.conv_q = ShortConv(h * dk, cfg.conv_kernel)
        self.conv_k = ShortConv(h * dk, cfg.conv_kernel)
        self.conv_v = ShortConv(h * dv, cfg.conv_kernel)

        self.W_alpha_down = nn.Linear(d, cfg.gate_rank, bias=False)
        self.W_alpha_up = nn.Linear(cfg.gate_rank, h * dk, bias=False)
        self.W_beta = nn.Linear(d, h, bias=True)

        self.W_g_down = nn.Linear(d, cfg.gate_rank, bias=False)
        self.W_g_up = nn.Linear(cfg.gate_rank, h * dv, bias=False)

        self.head_norm = RMSNorm(dv, eps=cfg.norm_eps)
        self.dropout = nn.Dropout(cfg.dropout)
        self.W_o = nn.Linear(h * dv, d, bias=False)

    def allocate_cache(self, batch_size: int, device: torch.device, dtype: torch.dtype) -> KDALayerCache:
        h, dk, dv = self.h, self.dk, self.dv
        return KDALayerCache(
            state=torch.zeros(batch_size, h, dk, dv, device=device, dtype=dtype),
            conv_q=self.conv_q.allocate_state(batch_size, device, dtype),
            conv_k=self.conv_k.allocate_state(batch_size, device, dtype),
            conv_v=self.conv_v.allocate_state(batch_size, device, dtype),
        )

    def _params_from_conv_outputs(
        self,
        x: torch.Tensor,
        q_conv: torch.Tensor,
        k_conv: torch.Tensor,
        v_conv: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        B, T, _ = x.shape
        h, dk, dv = self.h, self.dk, self.dv

        q = F.silu(q_conv).reshape(B, T, h, dk)
        k = F.silu(k_conv).reshape(B, T, h, dk)
        v = F.silu(v_conv).reshape(B, T, h, dv)

        q = F.normalize(q, p=2, dim=-1, eps=1e-8)
        k = F.normalize(k, p=2, dim=-1, eps=1e-8)

        # log alpha in [-10, 0]. exp(log_alpha) in [~4.5e-5, 1].
        log_alpha = F.logsigmoid(self.W_alpha_up(self.W_alpha_down(x)))
        log_alpha = log_alpha.reshape(B, T, h, dk).clamp(min=-10.0)
        beta = torch.sigmoid(self.W_beta(x)).reshape(B, T, h, 1)
        return q, k, v, log_alpha, beta

    def _finish_output(self, x: torch.Tensor, o: torch.Tensor) -> torch.Tensor:
        B, T, h, dv = o.shape
        o = self.head_norm(o)
        gate = torch.sigmoid(self.W_g_up(self.W_g_down(x))).reshape(B, T, h, dv)
        o = gate * o
        o = self.dropout(o)
        return self.W_o(o.reshape(B, T, h * dv))

    def forward(self, x: torch.Tensor, cache: Optional[KDALayerCache] = None, ) -> tuple[torch.Tensor, Optional[KDALayerCache]]:
        # Full sequence if cache is None; one-token recurrent step if cache is given.

        B, T, _ = x.shape
        q_raw = self.W_q(x)
        k_raw = self.W_k(x)
        v_raw = self.W_v(x)

        if cache is None:
            q_conv = self.conv_q.forward_full(q_raw)
            k_conv = self.conv_k.forward_full(k_raw)
            v_conv = self.conv_v.forward_full(v_raw)
            q, k, v, log_alpha, beta = self._params_from_conv_outputs(x, q_conv, k_conv, v_conv)
            o, _ = chunk_kda(q, k, v, log_alpha, beta, self.cfg.chunk_size)
            return self._finish_output(x, o), None

        assert T == 1 #Cached KDA forward is intended for one-token recurrent decoding
        q_conv, new_cq = self.conv_q.step(q_raw, cache.conv_q)
        k_conv, new_ck = self.conv_k.step(k_raw, cache.conv_k)
        v_conv, new_cv = self.conv_v.step(v_raw, cache.conv_v)
        q, k, v, log_alpha, beta = self._params_from_conv_outputs(x, q_conv, k_conv, v_conv)
        o, new_state = step_kda(q, k, v, log_alpha, beta, cache.state)
        new_cache = KDALayerCache(new_state, new_cq, new_ck, new_cv)
        return self._finish_output(x, o), new_cache

    def forward_with_cache(self, x: torch.Tensor) -> tuple[torch.Tensor, KDALayerCache]:
        # Parallel prefill that also extracts the recurrent cache after the sequence
        q_raw = self.W_q(x)
        k_raw = self.W_k(x)
        v_raw = self.W_v(x)

        q_conv = self.conv_q.forward_full(q_raw)
        k_conv = self.conv_k.forward_full(k_raw)
        v_conv = self.conv_v.forward_full(v_raw)

        q, k, v, log_alpha, beta = self._params_from_conv_outputs(x, q_conv, k_conv, v_conv)
        o, final_state = chunk_kda(q, k, v, log_alpha, beta, self.cfg.chunk_size)
        out = self._finish_output(x, o)

        cache = KDALayerCache(
            state=final_state.contiguous(),
            conv_q=self.conv_q.final_state_from_full_input(q_raw),
            conv_k=self.conv_k.final_state_from_full_input(k_raw),
            conv_v=self.conv_v.final_state_from_full_input(v_raw),
        )
        return out, cache


class NoPEMHALayer(nn.Module):

    # NoPE full multi-head attention.

    #It is not true latent-compression MLA implementation


    def __init__(self, cfg: ToyKimiConfig):
        super().__init__()
        d, h, dk, dv = cfg.d_model, cfg.n_heads, cfg.d_k, cfg.d_v
        self.h = h
        self.dk = dk
        self.dv = dv
        self.W_q = nn.Linear(d, h * dk, bias=False)
        self.W_k = nn.Linear(d, h * dk, bias=False)
        self.W_v = nn.Linear(d, h * dv, bias=False)
        self.W_o = nn.Linear(h * dv, d, bias=False)
        self.dropout_p = cfg.dropout

    def _qkv(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        B, T, _ = x.shape
        q = self.W_q(x).reshape(B, T, self.h, self.dk).transpose(1, 2)
        k = self.W_k(x).reshape(B, T, self.h, self.dk).transpose(1, 2)
        v = self.W_v(x).reshape(B, T, self.h, self.dv).transpose(1, 2)
        return q, k, v

    def forward(self, x: torch.Tensor, cache: Optional[MLACache] = None,) -> tuple[torch.Tensor, MLACache]:
        B, T, _ = x.shape
        q, k_new, v_new = self._qkv(x)

        if cache is None:
            k = k_new
            v = v_new
            is_causal = True
        else:
            k = torch.cat([cache.k, k_new], dim=2)
            v = torch.cat([cache.v, v_new], dim=2)
            is_causal = False

        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=is_causal,
        )
        out = self.W_o(attn_out.transpose(1, 2).reshape(B, T, self.h * self.dv))
        return out, MLACache(k=k, v=v)

    def forward_with_cache(self, x: torch.Tensor) -> tuple[torch.Tensor, MLACache]:
        return self.forward(x, cache=None)


class SwiGLUFFN(nn.Module):
    def __init__(self, cfg: ToyKimiConfig):
        super().__init__()
        d, ffn = cfg.d_model, cfg.ffn_dim
        self.W_gate = nn.Linear(d, ffn, bias=False)
        self.W_up = nn.Linear(d, ffn, bias=False)
        self.W_down = nn.Linear(ffn, d, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.silu(self.W_gate(x)) * self.W_up(x)
        x = self.dropout(x)
        return self.W_down(x)


class KimiLinearBlock(nn.Module):
    # Macro block: [KDA + FFN] x 3, then [NoPE full attention + FFN] x 1

    def __init__(self, cfg: ToyKimiConfig):
        super().__init__()
        self.kda_layers = nn.ModuleList([KDALayer(cfg) for _ in range(cfg.kda_per_block)])
        self.kda_norms = nn.ModuleList([RMSNorm(cfg.d_model, cfg.norm_eps) for _ in range(cfg.kda_per_block)])
        self.kda_ffns = nn.ModuleList([SwiGLUFFN(cfg) for _ in range(cfg.kda_per_block)])
        self.kda_ffn_norms = nn.ModuleList([RMSNorm(cfg.d_model, cfg.norm_eps) for _ in range(cfg.kda_per_block)])

        self.full_attn = NoPEMHALayer(cfg)
        self.full_attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.full_ffn = SwiGLUFFN(cfg)
        self.full_ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)

    def allocate_cache(self, batch_size: int, device: torch.device, dtype: torch.dtype) -> BlockCache:
        return BlockCache(
            kda_caches=[layer.allocate_cache(batch_size, device, dtype) for layer in self.kda_layers],
            mla_cache=None,
        )

    def forward(self, x: torch.Tensor, cache: Optional[BlockCache] = None) -> tuple[torch.Tensor, Optional[BlockCache]]:
        new_kda_caches: list[KDALayerCache] = []

        for i, (kda, norm, ffn, ffn_norm) in enumerate(
            zip(self.kda_layers, self.kda_norms, self.kda_ffns, self.kda_ffn_norms)
        ):
            layer_cache = cache.kda_caches[i] if cache is not None else None
            y, new_layer_cache = kda(norm(x), cache=layer_cache)
            x = x + y
            if cache is not None:
                assert new_layer_cache is not None
                new_kda_caches.append(new_layer_cache)
            x = x + ffn(ffn_norm(x))

        mla_cache = cache.mla_cache if cache is not None else None
        y, new_mla_cache = self.full_attn(self.full_attn_norm(x), cache=mla_cache)
        x = x + y
        x = x + self.full_ffn(self.full_ffn_norm(x))

        if cache is None:
            return x, None
        return x, BlockCache(kda_caches=new_kda_caches, mla_cache=new_mla_cache)

    def forward_with_cache(self, x: torch.Tensor) -> tuple[torch.Tensor, BlockCache]:
        caches: list[KDALayerCache] = []
        for kda, norm, ffn, ffn_norm in zip(
            self.kda_layers, self.kda_norms, self.kda_ffns, self.kda_ffn_norms
        ):
            y, layer_cache = kda.forward_with_cache(norm(x))
            x = x + y
            caches.append(layer_cache)
            x = x + ffn(ffn_norm(x))

        y, mla_cache = self.full_attn.forward_with_cache(self.full_attn_norm(x))
        x = x + y
        x = x + self.full_ffn(self.full_ffn_norm(x))
        return x, BlockCache(kda_caches=caches, mla_cache=mla_cache)


# Full model


class ToyKimiLinear(nn.Module):
    def __init__(self, cfg: ToyKimiConfig):
        super().__init__()
        cfg.validate()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([KimiLinearBlock(cfg) for _ in range(cfg.n_blocks)])
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        self.apply(self._init_weights)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

    def _init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def allocate_cache(self, batch_size: int, device: torch.device, dtype: torch.dtype) -> list[BlockCache]:
        return [block.allocate_cache(batch_size, device, dtype) for block in self.blocks]

    def forward(self, input_ids: torch.Tensor, targets: Optional[torch.Tensor] = None, caches: Optional[list[BlockCache]] = None,) -> tuple[torch.Tensor, Optional[torch.Tensor], Optional[list[BlockCache]]]:
        x = self.tok_emb(input_ids)
        x = self.drop(x)

        new_caches: Optional[list[BlockCache]] = [] if caches is not None else None
        for i, block in enumerate(self.blocks):
            block_cache = caches[i] if caches is not None else None
            x, new_block_cache = block(x, cache=block_cache)
            if new_caches is not None:
                assert new_block_cache is not None
                new_caches.append(new_block_cache)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, new_caches

    @torch.no_grad()
    def prefill(self, input_ids: torch.Tensor) -> tuple[torch.Tensor, list[BlockCache]]:
        # Full-sequence prefill that extracts caches for recurrent decoding
        x = self.tok_emb(input_ids)
        x = self.drop(x)
        caches: list[BlockCache] = []
        for block in self.blocks:
            x, cache = block.forward_with_cache(x)
            caches.append(cache)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits, caches

    @torch.no_grad()
    def step(self, input_id_t: torch.Tensor, caches: list[BlockCache]) -> tuple[torch.Tensor, list[BlockCache]]:
        #One-token recurrent decode step. input_id_t: [B]
        logits, _, new_caches = self.forward(input_id_t[:, None], targets=None, caches=caches)
        assert new_caches is not None
        return logits[:, -1, :], new_caches

    @torch.no_grad()
    def generate_full_prefix(self,input_ids: torch.Tensor, max_new_tokens: int = 128, temperature: float = 0.8, top_k: int = 40,) -> torch.Tensor:
        self.eval()
        out = input_ids
        for _ in range(max_new_tokens):
            idx = out[:, -self.cfg.max_seq_len :]
            logits, _, _ = self.forward(idx)
            next_logits = logits[:, -1, :] / max(temperature, 1e-8)
            if top_k > 0:
                values, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
                next_logits = next_logits.masked_fill(next_logits < values[:, [-1]], -float("inf"))
            probs = F.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            out = torch.cat([out, next_id], dim=1)
        return out

    @torch.no_grad()
    def generate_recurrent(self, input_ids: torch.Tensor, max_new_tokens: int = 128, temperature: float = 0.8, top_k: int = 40, optimized_prefill: bool = True,) -> torch.Tensor:
        self.eval()
        B = input_ids.size(0)
        device = input_ids.device
        dtype = self.tok_emb.weight.dtype

        if optimized_prefill:
            prefill_logits, caches = self.prefill(input_ids)
            logits = prefill_logits[:, -1, :]
        else:
            caches = self.allocate_cache(B, device, dtype)
            logits = None
            for t in range(input_ids.size(1)):
                logits, caches = self.step(input_ids[:, t], caches)
            assert logits is not None

        out = input_ids
        for _ in range(max_new_tokens):
            next_logits = logits / max(temperature, 1e-8)
            if top_k > 0:
                values, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
                next_logits = next_logits.masked_fill(next_logits < values[:, [-1]], -float("inf"))
            probs = F.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).squeeze(-1)
            out = torch.cat([out, next_id[:, None]], dim=1)
            logits, caches = self.step(next_id, caches)
        return out

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int = 128,
        temperature: float = 0.8,
        top_k: int = 40,
        use_recurrent: bool = True,
        optimized_prefill: bool = True,
    ) -> torch.Tensor:
        if use_recurrent:
            return self.generate_recurrent(
                input_ids,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_k=top_k,
                optimized_prefill=optimized_prefill,
            )
        return self.generate_full_prefix(input_ids, max_new_tokens, temperature, top_k)

    def num_parameters(self, only_trainable: bool = True) -> int:
        return sum(
            p.numel() for p in self.parameters() if (not only_trainable or p.requires_grad)
        )


In [3]:
# Mamba model components

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Compute in fp32 for stability, return in the input dtype for AMP friendliness.
        dtype = x.dtype
        x_float = x.float()
        rms = torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        y = x_float * rms * self.weight.float()
        return y.to(dtype)


def segsum(x: torch.Tensor) -> torch.Tensor:
    # Stable segment-sum helper for a lower-triangular 1-semiseparable mask.

    T = x.size(-1)
    cumsum = x.cumsum(dim=-1)
    diff = cumsum[..., :, None] - cumsum[..., None, :]
    mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
    return diff.masked_fill(~mask, -float("inf"))


def ssd(
    X: torch.Tensor,      # [B, T, H, P]
    A: torch.Tensor,      # [B, T, H], log-decay values, usually negative
    B: torch.Tensor,      # [B, T, H, N]
    C: torch.Tensor,      # [B, T, H, N]
    block_len: int = 64,
    initial_states: Optional[torch.Tensor] = None,  # [B, 1, H, P, N]
) -> tuple[torch.Tensor, torch.Tensor]:

    # Block-decomposed SSD forward pass.

    #Returns:
    #   Y: [B, T, H, P]
    #   final_state: [B, H, P, N]

    batch, T_orig, H, P = X.shape
    N = B.shape[-1]
    Q = block_len

    assert A.shape == (batch, T_orig, H)
    assert B.shape == (batch, T_orig, H, N)
    assert C.shape == (batch, T_orig, H, N)

    pad = (Q - T_orig % Q) % Q
    if pad > 0:
        X = F.pad(X, (0, 0, 0, 0, 0, pad))
        A = F.pad(A, (0, 0, 0, pad))
        B = F.pad(B, (0, 0, 0, 0, 0, pad))
        C = F.pad(C, (0, 0, 0, 0, 0, pad))

    T = T_orig + pad
    n_chunks = T // Q

    X = X.reshape(batch, n_chunks, Q, H, P)
    A = A.reshape(batch, n_chunks, Q, H)
    B = B.reshape(batch, n_chunks, Q, H, N)
    C = C.reshape(batch, n_chunks, Q, H, N)

    # [B, H, C, Q]
    A = A.permute(0, 3, 1, 2).contiguous()
    A_cumsum = A.cumsum(dim=-1)

    # Diagonal blocks: intra-chunk quadratic part
    L = segsum(A).exp()  # [B, H, C, Q, Q]
    G = torch.einsum("bclhn,bcshn->bhcls", C, B)
    Y_diag = torch.einsum("bhcls,bcshp->bclhp", L * G, X)

    # Right factors: input positions to per-chunk state
    decay_states = (A_cumsum[..., -1:] - A_cumsum).exp()  # [B, H, C, Q]
    chunk_states = torch.einsum("bclhn,bhcl,bclhp->bchpn", B, decay_states, X)

    # Center factors: chunk-to-chunk recurrence/scan
    if initial_states is None:
        initial_states = chunk_states.new_zeros(batch, 1, H, P, N)
    else:
        assert initial_states.shape == (batch, 1, H, P, N)

    states_with_initial = torch.cat([initial_states, chunk_states], dim=1)

    # A_chunk_total[:, :, 0] corresponds to the initial state boundary.
    A_chunk_total = F.pad(A_cumsum[..., -1], (1, 0))  # [B, H, C + 1]
    decay_chunk = segsum(A_chunk_total).exp()         # [B, H, C + 1, C + 1]
    scanned_states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states_with_initial)

    states = scanned_states[:, :-1]       # [B, C, H, P, N]
    final_state = scanned_states[:, -1]   # [B, H, P, N]

    # Left factors: per-chunk state to outputs.
    state_decay_out = A_cumsum.exp()  # [B, H, C, Q]
    Y_off = torch.einsum("bclhn,bchpn,bhcl->bclhp", C, states, state_decay_out)

    Y = (Y_diag + Y_off).reshape(batch, T, H, P)
    if pad > 0:
        Y = Y[:, :T_orig]

    return Y, final_state


@torch.no_grad()
def ssd_recurrent_naive(X: torch.Tensor, A: torch.Tensor, B: torch.Tensor, C: torch.Tensor,) -> torch.Tensor:
    #Slow recurrent SSD reference implementation
    #Use only for correctness tests

    batch, T, H, P = X.shape
    N = B.shape[-1]
    state = X.new_zeros(batch, H, P, N)
    outputs = []

    for t in range(T):
        decay = torch.exp(A[:, t]).view(batch, H, 1, 1)
        state = decay * state + torch.einsum("bhn,bhp->bhpn", B[:, t], X[:, t])
        y_t = torch.einsum("bhn,bhpn->bhp", C[:, t], state)
        outputs.append(y_t)

    return torch.stack(outputs, dim=1)


def test_ssd_correctness(device: torch.device, dtype: torch.dtype = torch.float32) -> None:
    #Verifies that block SSD matches the naive recurrence for several sequence lengths, including lengths not divisible by block_len

    torch.manual_seed(0)
    test_cases = [
        dict(T=1, block_len=4),
        dict(T=7, block_len=4),
        dict(T=16, block_len=8),
        dict(T=17, block_len=8),
        dict(T=33, block_len=16),
    ]

    for case in test_cases:
        batch, T, H, P, N = 2, case["T"], 3, 5, 7
        block_len = case["block_len"]

        X = torch.randn(batch, T, H, P, device=device, dtype=dtype)
        # keep A negative and not too large in magnitude to avoid numerical extremes.
        A = -0.1 * torch.rand(batch, T, H, device=device, dtype=dtype)
        B_ = torch.randn(batch, T, H, N, device=device, dtype=dtype)
        C_ = torch.randn(batch, T, H, N, device=device, dtype=dtype)

        Y_fast, _ = ssd(X, A, B_, C_, block_len=block_len)
        Y_slow = ssd_recurrent_naive(X, A, B_, C_)
        max_err = (Y_fast.float() - Y_slow.float()).abs().max().item()

        tolerance = 2e-4 if dtype == torch.float32 else 5e-2
        if max_err > tolerance:
            raise AssertionError(
                f"SSD correctness failed for T={T}, block_len={block_len}: "
                f"max_err={max_err:.6g}, tolerance={tolerance:.6g}"
            )

    print("SSD correctness test passed.")


@torch.no_grad()
def test_recurrent_matches_full_forward(device: torch.device) -> None:
    # Verifies that recurrent prefill gives approximately the same logits as the full-sequence forward path.

    torch.manual_seed(123)
    cfg = ToyMamba2Config.quarter(max_seq_len=32, block_len=8, dropout=0.0)
    model = ToyMamba2(cfg).to(device)
    model.eval()

    input_ids = torch.randint(0, cfg.vocab_size, (2, 19), device=device)
    logits_full, _ = model(input_ids)

    caches = model.allocate_cache(
        batch_size=input_ids.size(0),
        device=device,
        dtype=model.tok_emb.weight.dtype,
    )
    logits_step = None
    for t in range(input_ids.size(1)):
        logits_step, caches = model.step(input_ids[:, t], caches)

    assert logits_step is not None
    max_err = (logits_full[:, -1].float() - logits_step.float()).abs().max().item()
    tolerance = 5e-4
    if max_err > tolerance:
        raise AssertionError(
            f"Recurrent/full-forward mismatch: max_err={max_err:.6g}, "
            f"tolerance={tolerance:.6g}"
        )

    print("Recurrent step-by-step prefill matches full forward test passed.")


@torch.no_grad()
def test_optimized_prefill_matches_full_and_step(device: torch.device) -> None:
    # Verifies optimized parallel prefill cache extraction

    #Test 1: prefill(prompt) logits match normal full forward(prompt) logits
    # Test 2: prefill(prompt) + step(next_token) matches full forward(prompt + next_token) on the final token logits

    #Trying to catch off-by-one errors in conv_state and SSM final_state extraction

    torch.manual_seed(456)
    cfg = ToyMamba2Config.quarter(max_seq_len=64, block_len=8, dropout=0.0)
    model = ToyMamba2(cfg).to(device)
    model.eval()

    prompt = torch.randint(0, cfg.vocab_size, (2, 19), device=device)
    next_token = torch.randint(0, cfg.vocab_size, (2,), device=device)
    prompt_plus_one = torch.cat([prompt, next_token[:, None]], dim=1)

    logits_full_prompt, _ = model(prompt)
    logits_prefill, caches = model.prefill(prompt)

    err_prefill = (logits_full_prompt.float() - logits_prefill.float()).abs().max().item()
    tolerance_prefill = 5e-4
    if err_prefill > tolerance_prefill:
        raise AssertionError(
            f"Optimized prefill logits mismatch: max_err={err_prefill:.6g}, "
            f"tolerance={tolerance_prefill:.6g}"
        )

    logits_full_plus_one, _ = model(prompt_plus_one)
    logits_next_step, _ = model.step(next_token, caches)

    err_next = (logits_full_plus_one[:, -1].float() - logits_next_step.float()).abs().max().item()
    tolerance_next = 5e-4
    if err_next > tolerance_next:
        raise AssertionError(
            f"Optimized prefill + step mismatch: max_err={err_next:.6g}, "
            f"tolerance={tolerance_next:.6g}"
        )

    print("Optimized parallel prefill cache test passed.")


@dataclass
class Mamba2LayerCache:
    conv_state: torch.Tensor  # [B, conv_channels, conv_kernel - 1]
    ssm_state: torch.Tensor   # [B, H, P, N]


class Mamba2Block(nn.Module):
    def __init__(self, cfg: ToyMamba2Config):
        super().__init__()
        cfg.validate()

        d = cfg.d_model
        E = cfg.d_inner
        H = cfg.n_heads
        N = cfg.d_state
        P = cfg.d_head
        G = cfg.n_groups

        self.H = H
        self.P = P
        self.N = N
        self.G = G
        self.E = E
        self.block_len = cfg.block_len
        self.dropout_p = cfg.dropout

        conv_channels = E + 2 * G * N
        proj_dim = E + conv_channels + H

        self.in_proj = nn.Linear(d, proj_dim, bias=False)

        self.conv1d = nn.Conv1d(
            conv_channels,
            conv_channels,
            kernel_size=cfg.conv_dim,
            padding=cfg.conv_dim - 1,
            groups=conv_channels,
            bias=True,
        )

        # Scalar-identity A per head. A = -exp(A_log)
        self.A_log = nn.Parameter(torch.log(torch.empty(H).uniform_(1, 16)))

        # Inverse softplus initialization for dt in approximately [0.001, 0.1]
        dt_init = torch.exp(
            torch.rand(H) * (math.log(0.1) - math.log(0.001)) + math.log(0.001)
        ).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt_init + torch.log(-torch.expm1(-dt_init)))

        # Per-head skip/feed-through
        self.D = nn.Parameter(torch.ones(H))

        # Mamba-2-style order:
        #     y = y * silu(z)
        #     y = norm(y)
        #     out = out_proj(y)
        self.norm = RMSNorm(E, eps=cfg.norm_eps)
        self.dropout = nn.Dropout(cfg.dropout)
        self.out_proj = nn.Linear(E, d, bias=False)

        self.conv_channels = conv_channels
        self.conv_kernel_size = cfg.conv_dim
        self.proj_splits = [E, conv_channels, H]
        self.conv_splits = [E, G * N, G * N]

    def allocate_cache(self, batch_size: int, device: torch.device, dtype: torch.dtype,) -> Mamba2LayerCache:
        # Allocate recurrent inference cache for one Mamba block

        return Mamba2LayerCache(
            conv_state=torch.zeros(
                batch_size,
                self.conv_channels,
                self.conv_kernel_size - 1,
                device=device,
                dtype=dtype,
            ),
            ssm_state=torch.zeros(
                batch_size,
                self.H,
                self.P,
                self.N,
                device=device,
                dtype=dtype,
            ),
        )

    def _project_and_conv_step(
        self,
        u_t: torch.Tensor,  # [B, d_model]
        cache: Mamba2LayerCache,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, Mamba2LayerCache]:
        # One-token projection + causal depthwise conv.


        batch = u_t.size(0)
        H, P, N, G = self.H, self.P, self.N, self.G

        z, xBC, dt = self.in_proj(u_t).split(self.proj_splits, dim=-1)

        # Manual one-step causal depthwise conv.
        # Training path uses Conv1d with padding=K-1 and then crops to [:T]
        # For one-step inference, the equivalent input window is the cached K-1
        # previous xBC vectors plus the current xBC vector.
        xBC_ch = xBC.unsqueeze(-1)  # [B, conv_channels, 1]
        if self.conv_kernel_size > 1:
            conv_window = torch.cat([cache.conv_state, xBC_ch], dim=-1)
        else:
            conv_window = xBC_ch

        # Conv1d is depthwise: weight [conv_channels, 1, K].
        weight = self.conv1d.weight[:, 0, :].to(conv_window.dtype)  # [conv_channels, K]
        xBC_conv = (conv_window * weight.unsqueeze(0)).sum(dim=-1)
        if self.conv1d.bias is not None:
            xBC_conv = xBC_conv + self.conv1d.bias.to(xBC_conv.dtype)
        xBC_conv = F.silu(xBC_conv)

        if self.conv_kernel_size > 1:
            cache.conv_state = conv_window[:, :, 1:].contiguous()

        x, b_flat, c_flat = xBC_conv.split(self.conv_splits, dim=-1)
        x = x.reshape(batch, H, P)

        b = b_flat.reshape(batch, G, N)
        c = c_flat.reshape(batch, G, N)
        if G != H:
            repeats = H // G
            b = b.repeat_interleave(repeats, dim=1)
            c = c.repeat_interleave(repeats, dim=1)

        return z, x, b, c, dt, cache

    def step(self,
        u_t: torch.Tensor,  # [B, d_model]
        cache: Mamba2LayerCache,
    ) -> tuple[torch.Tensor, Mamba2LayerCache]:
        # Recurrent one-token inference step.

        # equivalent to the full-sequence path under this toy SSD parameterization, up to numerical tolerance.
        batch = u_t.size(0)
        H = self.H

        z, x, b, c, dt, cache = self._project_and_conv_step(u_t, cache)

        A = -torch.exp(self.A_log.float()).to(dt.dtype)  # [H]
        dt_pos = F.softplus(dt + self.dt_bias.to(dt.dtype))
        a = dt_pos * A.view(1, H)                       # [B, H]
        decay = torch.exp(a).view(batch, H, 1, 1)

        cache.ssm_state = (
            decay * cache.ssm_state
            + torch.einsum("bhn,bhp->bhpn", b, x)
        )

        y = torch.einsum("bhn,bhpn->bhp", c, cache.ssm_state)
        y = y + self.D.to(y.dtype).view(1, H, 1) * x
        y = y.reshape(batch, self.E)

        y = y * F.silu(z)
        y = self.norm(y)
        y = self.out_proj(y)
        return y, cache

    def _forward_impl(self, u: torch.Tensor, return_cache: bool = False,) -> torch.Tensor | tuple[torch.Tensor, Mamba2LayerCache]:
        #Full-sequence block forward.

        # If return_cache=True, also returns the recurrent cache corresponding to the state after consuming the whole sequence. This enables optimized
        #generation: parallel/full SSD prefill + recurrent one-token decoding

        batch, T, _ = u.shape
        H, P, N, G, E = self.H, self.P, self.N, self.G, self.E

        z, raw_xBC, dt = self.in_proj(u).split(self.proj_splits, dim=-1)

        # Save the raw pre-conv xBC tail for recurrent conv_state.
        # The recurrent conv step needs the last K-1 raw projected xBC vectors
        if return_cache and self.conv_kernel_size > 1:
            tail_len = self.conv_kernel_size - 1
            if T >= tail_len:
                conv_state = raw_xBC[:, -tail_len:, :].transpose(1, 2).contiguous()
            else:
                # Left-pad with zeros if the prompt is shorter than K-1
                pad_len = tail_len - T
                zeros = raw_xBC.new_zeros(batch, pad_len, self.conv_channels)
                conv_state = torch.cat([zeros, raw_xBC], dim=1).transpose(1, 2).contiguous()
        elif return_cache:
            conv_state = raw_xBC.new_zeros(batch, self.conv_channels, 0)

        # Causal depthwise conv. Conv1d padding adds left and right padding; crop to T
        xBC = self.conv1d(raw_xBC.transpose(1, 2))[..., :T].transpose(1, 2)
        xBC = F.silu(xBC)
        x, b_flat, c_flat = xBC.split(self.conv_splits, dim=-1)

        x = x.reshape(batch, T, H, P)

        b = b_flat.reshape(batch, T, G, N)
        c = c_flat.reshape(batch, T, G, N)

        if G != H:
            repeats = H // G
            b = b.repeat_interleave(repeats, dim=2)
            c = c.repeat_interleave(repeats, dim=2)

        # Toy SSD convention: pass log-decay A into ssd(); ssd() exponentiates it
        A = -torch.exp(self.A_log.float()).to(dt.dtype)  # [H]
        dt_pos = F.softplus(dt + self.dt_bias.to(dt.dtype))
        a = dt_pos * A.view(1, 1, H)                     # [B, T, H], negative

        y, final_state = ssd(x, a, b, c, self.block_len)

        # D skip connection.
        y = y + self.D.to(y.dtype).view(1, 1, H, 1) * x

        y = y.reshape(batch, T, E)

        y = y * F.silu(z)
        y = self.norm(y)
        y = self.dropout(y)
        out = self.out_proj(y)

        if return_cache:
            cache = Mamba2LayerCache(
                conv_state=conv_state,
                ssm_state=final_state.contiguous(),
            )
            return out, cache
        return out

    def forward_with_cache(self, u: torch.Tensor) -> tuple[torch.Tensor, Mamba2LayerCache]:
        # Parallel/full-sequence prefill for one block.

        # Returns the full block output and a recurrent cache that can be used for the next token.

        out, cache = self._forward_impl(u, return_cache=True)
        return out, cache

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        return self._forward_impl(u, return_cache=False)


class ToyMamba2(nn.Module):
    def __init__(self, cfg: ToyMamba2Config):
        super().__init__()
        cfg.validate()
        self.cfg = cfg

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList(
            [
                nn.ModuleDict(
                    {
                        "norm": RMSNorm(cfg.d_model, cfg.norm_eps),
                        "mamba": Mamba2Block(cfg),
                    }
                )
                for _ in range(cfg.n_layers)
            ]
        )
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        self.apply(self._init_weights)

        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

    def _init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def allocate_cache(self, batch_size: int, device: torch.device, dtype: torch.dtype,) -> list[Mamba2LayerCache]:
        return [
            layer["mamba"].allocate_cache(batch_size, device, dtype)
            for layer in self.layers
        ]

    @torch.no_grad()
    def prefill(self,input_ids: torch.Tensor,) -> tuple[torch.Tensor, list[Mamba2LayerCache]]:
        # Kinda optimized prompt prefill.

        # Runs the prompt through the normal full-sequence SSD path, which is parallel over sequence length, while extracting recurrent caches for all
        #layers. Use before recurrent token-by-token decoding.


        x = self.tok_emb(input_ids)
        x = self.drop(x)
        caches: list[Mamba2LayerCache] = []

        for layer in self.layers:
            residual = x
            x_norm = layer["norm"](x)
            y, cache = layer["mamba"].forward_with_cache(x_norm)
            x = residual + y
            caches.append(cache)

        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits, caches

    @torch.no_grad()
    def step(
        self,
        input_id_t: torch.Tensor,  # [B]
        caches: list[Mamba2LayerCache],
    ) -> tuple[torch.Tensor, list[Mamba2LayerCache]]:
        # One autoregressive recurrent step.

        x = self.tok_emb(input_id_t)  # [B, d_model]
        new_caches: list[Mamba2LayerCache] = []

        for layer, cache in zip(self.layers, caches):
            residual = x
            x_norm = layer["norm"](x)
            y, new_cache = layer["mamba"].step(x_norm, cache)
            x = residual + y
            new_caches.append(new_cache)

        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits, new_caches

    def forward(self,input_ids: torch.Tensor,targets: Optional[torch.Tensor] = None,) -> tuple[torch.Tensor, Optional[torch.Tensor]]:
        x = self.tok_emb(input_ids)
        x = self.drop(x)

        for layer in self.layers:
            x = x + layer["mamba"](layer["norm"](x))

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
            )
        return logits, loss

    @torch.no_grad()
    def generate_full_prefix(self, input_ids: torch.Tensor, max_new_tokens: int = 128, temperature: float = 0.8, top_k: int = 40,) -> torch.Tensor:
        """
        Simple generation by full-prefix recomputation.

        Correct but not optimized. This does not use recurrent SSM states.
        Use mainly for debugging and as a reference against recurrent generation.
        """
        self.eval()
        for _ in range(max_new_tokens):
            idx = input_ids[:, -self.cfg.max_seq_len :]
            logits, _ = self.forward(idx)
            logits = logits[:, -1, :] / max(temperature, 1e-8)

            if top_k > 0:
                values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits = logits.masked_fill(logits < values[:, [-1]], -float("inf"))

            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_id], dim=1)

        return input_ids

    @torch.no_grad()
    def generate_recurrent(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int = 128,
        temperature: float = 0.8,
        top_k: int = 40,
        optimized_prefill: bool = True,
    ) -> torch.Tensor:
        # Recurrent generation using conv_state + ssm_state caches.


        # Set optimized_prefill=False to use the slower step-by-step prefill path

        self.eval()
        batch_size = input_ids.size(0)
        device = input_ids.device
        cache_dtype = self.tok_emb.weight.dtype

        if optimized_prefill:
            prefill_logits, caches = self.prefill(input_ids)
            logits = prefill_logits[:, -1, :]
        else:
            caches = self.allocate_cache(batch_size, device, cache_dtype)
            logits = None
            for t in range(input_ids.size(1)):
                logits, caches = self.step(input_ids[:, t], caches)
            assert logits is not None

        out = input_ids

        for _ in range(max_new_tokens):
            next_logits = logits / max(temperature, 1e-8)
            if top_k > 0:
                values, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
                next_logits = next_logits.masked_fill(next_logits < values[:, [-1]], -float("inf"))

            probs = F.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).squeeze(-1)
            out = torch.cat([out, next_id[:, None]], dim=1)
            logits, caches = self.step(next_id, caches)

        return out

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int = 128,
        temperature: float = 0.8,
        top_k: int = 40,
        use_recurrent: bool = True,
        optimized_prefill: bool = True,
    ) -> torch.Tensor:
        if use_recurrent:
            return self.generate_recurrent(
                input_ids,
                max_new_tokens,
                temperature,
                top_k,
                optimized_prefill=optimized_prefill,
            )
        return self.generate_full_prefix(input_ids, max_new_tokens, temperature, top_k)

    def num_parameters(self, only_trainable: bool = True) -> int:
        return sum(
            p.numel()
            for p in self.parameters()
            if (not only_trainable or p.requires_grad)
        )

In [4]:
# Transformer Model

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dtype = x.dtype
        x_float = x.float()
        rms = torch.rsqrt(x_float.pow(2).mean(-1, keepdim=True) + self.eps)
        out = x_float * rms * self.weight.float()
        return out.to(dtype)


class RotaryEmbedding(nn.Module):
    # RoPE cache that can safely extend if a longer sequence is seen.

    def __init__(self, dim: int, max_seq_len: int = 4096, base: float = 10000.0):
        super().__init__()
        self.dim = dim
        self.base = base
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.register_buffer("cos_cached", torch.empty(0), persistent=False)
        self.register_buffer("sin_cached", torch.empty(0), persistent=False)
        self._build_cache(max_seq_len, device=inv_freq.device, dtype=torch.float32)

    def _build_cache(self, seq_len: int, device: torch.device, dtype: torch.dtype) -> None:
        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq.to(device=device))
        emb = torch.cat([freqs, freqs], dim=-1)
        self.cos_cached = emb.cos().to(dtype=dtype)
        self.sin_cached = emb.sin().to(dtype=dtype)

    def forward(
        self,
        seq_len: int,
        device: torch.device,
        dtype: torch.dtype,
        offset: int = 0,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        needed = offset + seq_len
        if self.cos_cached.numel() == 0 or needed > self.cos_cached.size(0):
            new_len = max(needed, 2 * max(1, self.cos_cached.size(0)))
            self._build_cache(new_len, device=device, dtype=torch.float32)

        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(
    q: torch.Tensor,
    k: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    # q, k: [B, H, T, D]
    # cos, sin: [T, D]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    q = q * cos + rotate_half(q) * sin
    k = k * cos + rotate_half(k) * sin
    return q, k


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: ToyTransformerConfig):
        super().__init__()
        d, h, dh = cfg.d_model, cfg.n_heads, cfg.d_head
        self.h = h
        self.dh = dh

        self.W_q = nn.Linear(d, h * dh, bias=False)
        self.W_k = nn.Linear(d, h * dh, bias=False)
        self.W_v = nn.Linear(d, h * dh, bias=False)
        self.W_o = nn.Linear(h * dh, d, bias=False)

        self.rope = RotaryEmbedding(dh, max_seq_len=cfg.max_seq_len)
        self.dropout = cfg.dropout

    def forward(
        self,
        x: torch.Tensor,
        kv_cache: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> tuple[torch.Tensor, Optional[tuple[torch.Tensor, torch.Tensor]]]:
        B, T, _ = x.shape
        h, dh = self.h, self.dh

        q = self.W_q(x).view(B, T, h, dh).transpose(1, 2)  # [B,H,T,D]
        k = self.W_k(x).view(B, T, h, dh).transpose(1, 2)
        v = self.W_v(x).view(B, T, h, dh).transpose(1, 2)

        past_len = 0 if kv_cache is None else kv_cache[0].size(2)
        cos, sin = self.rope(
            seq_len=T,
            device=x.device,
            dtype=q.dtype,
            offset=past_len,
        )
        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)

        new_cache = (k, v) if use_cache else None

        # Full sequence training/prefill: causal mask.
        # Cached decoding: current token attends to all cached keys, so no causal mask needed.
        is_causal = kv_cache is None and T > 1

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )
        out = out.transpose(1, 2).reshape(B, T, h * dh)
        return self.W_o(out), new_cache


class SwiGLUFFN(nn.Module):
    def __init__(self, cfg: ToyTransformerConfig):
        super().__init__()
        d, ffn = cfg.d_model, cfg.ffn_dim
        self.W_gate = nn.Linear(d, ffn, bias=False)
        self.W_up = nn.Linear(d, ffn, bias=False)
        self.W_down = nn.Linear(ffn, d, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.silu(self.W_gate(x)) * self.W_up(x)
        x = self.W_down(x)
        return self.dropout(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: ToyTransformerConfig):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = CausalSelfAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLUFFN(cfg)

    def forward(
        self,
        x: torch.Tensor,
        kv_cache: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> tuple[torch.Tensor, Optional[tuple[torch.Tensor, torch.Tensor]]]:
        attn_out, new_cache = self.attn(
            self.attn_norm(x),
            kv_cache=kv_cache,
            use_cache=use_cache,
        )
        x = x + attn_out
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_cache


class ToyRoPETransformer(nn.Module):
    def __init__(self, cfg: ToyTransformerConfig):
        super().__init__()
        cfg.validate()
        self.cfg = cfg

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        self.apply(self._init_weights)

        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

    def _init_weights(self, m: nn.Module) -> None:
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(
        self,
        input_ids: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
        kv_caches: Optional[list[Optional[tuple[torch.Tensor, torch.Tensor]]]] = None,
        use_cache: bool = False,
    ) -> tuple[
        torch.Tensor,
        Optional[torch.Tensor],
        Optional[list[Optional[tuple[torch.Tensor, torch.Tensor]]]],
    ]:
        x = self.tok_emb(input_ids)

        new_caches = [] if use_cache else None
        if kv_caches is None:
            kv_caches = [None] * len(self.layers)

        for layer, cache in zip(self.layers, kv_caches):
            x, new_cache = layer(x, kv_cache=cache, use_cache=use_cache)
            if use_cache:
                new_caches.append(new_cache)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
            )

        return logits, loss, new_caches

    @torch.no_grad()
    def generate(self, input_ids: torch.Tensor, max_new_tokens: int = 128, temperature: float = 0.8, top_k: int = 40, use_cache: bool = True,) -> torch.Tensor:
        self.eval()

        if not use_cache:
            return self._generate_naive(
                input_ids=input_ids,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_k=top_k,
            )

        idx = input_ids[:, -self.cfg.max_seq_len:]
        logits, _, caches = self.forward(idx, use_cache=True)
        next_logits = logits[:, -1, :] / max(temperature, 1e-8)

        generated = input_ids
        for _ in range(max_new_tokens):
            if top_k > 0:
                values, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
                next_logits = next_logits.masked_fill(next_logits < values[:, [-1]], -float("inf"))

            probs = F.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            generated = torch.cat([generated, next_id], dim=1)

            # If cache grows past max_seq_len, recompute recent context
            if generated.size(1) > self.cfg.max_seq_len:
                idx = generated[:, -self.cfg.max_seq_len:]
                logits, _, caches = self.forward(idx, use_cache=True)
            else:
                logits, _, caches = self.forward(next_id, kv_caches=caches, use_cache=True)

            next_logits = logits[:, -1, :] / max(temperature, 1e-8)

        return generated

    @torch.no_grad()
    def _generate_naive(self, input_ids: torch.Tensor, max_new_tokens: int, temperature: float, top_k: int,) -> torch.Tensor:
        self.eval()
        for _ in range(max_new_tokens):
            idx = input_ids[:, -self.cfg.max_seq_len:]
            logits, _, _ = self.forward(idx)
            logits = logits[:, -1, :] / max(temperature, 1e-8)

            if top_k > 0:
                values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits = logits.masked_fill(logits < values[:, [-1]], -float("inf"))

            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_id], dim=1)

        return input_ids

    def num_parameters(self, only_trainable: bool = True) -> int:
        return sum(
            p.numel()
            for p in self.parameters()
            if (not only_trainable or p.requires_grad)
        )


In [5]:
# Evaluator for models
# Config

@dataclass
class ColabEvalConfig:
    TRANSFORMER_CKPT: Optional[str] = "/content/drive/MyDrive/models/transformer_toy2/checkpoints/toy_rope_transformer_half_param_matched_tinystories_20260503_085746/best.pt"
    MAMBA_CKPT: Optional[str] = "/content/drive/MyDrive/models/mamba2_toy/checkpoints/toy_mamba2_half_tinystories_20260501_090228/best.pt"
    KIMI_CKPT: Optional[str] = "/content/drive/MyDrive/models/kimi_linear_toy/checkpoints/toy_kimi_linear_half_tinystories_20260501_124445/best.pt"

    OUT_DIR: str = "/content/drive/MyDrive/models/eval_outputs"

    DEVICE: str = "auto"

    DATASET: str = "tinystories"
    MAX_TOKENS: Optional[int] = None

    VAL_MONITOR_FRACTION: float = 0.02
    VAL_ABLATION_FRACTION: float = 0.015
    TEST_REPLICATION_FRACTION: float = 0.015

    VAL_MONITOR_MAX_TOKENS: Optional[int] = 2_000_000
    VAL_ABLATION_MAX_TOKENS: Optional[int] = 1_000_000
    TEST_REPLICATION_MAX_TOKENS: Optional[int] = 1_000_000

    EVAL_BATCH_SIZE: int = 24
    EVAL_BATCHES: int = 200
    NUM_WORKERS: int = 2

    GENERATION_PROMPTS: int = 20
    GENERATION_NEW_TOKENS: int = 100
    GENERATION_MODES: tuple[str, ...] = ("auto", "full_prefix")

    PROBE_NEW_TOKENS: int = 8

    RUN_LM_EVAL: bool = True
    RUN_GENERATION_SPEED: bool = True
    RUN_PROBES: bool = True

    STRICT_LOAD: bool = True


CFG = ColabEvalConfig()

In [6]:
# Utilities

ARCH_CLASSES = {
    "transformer": {
        "config": "ToyTransformerConfig",
        "model": "ToyRoPETransformer",
    },
    "mamba": {
        "config": "ToyMamba2Config",
        "model": "ToyMamba2",
    },
    "kimi": {
        "config": "ToyKimiConfig",
        "model": "ToyKimiLinear",
    },
}


def require_global_class(name: str):
    if name not in globals():
        raise NameError(
            f"Missing class `{name}` in notebook mem"
            f"Run/copy the full model definition cell that defines `{name}` before evaluation."
        )
    return globals()[name]


def choose_device(device_arg: str) -> torch.device:
    if device_arg == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(device_arg)


def choose_amp_dtype(device: torch.device) -> tuple[torch.dtype, bool]:
    if device.type == "cuda":
        return (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16), True
    return torch.float32, False


def sync_if_cuda(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def gpu_stats(device: torch.device) -> dict[str, float]:
    if device.type != "cuda":
        return {}
    return {
        "gpu_mem_allocated_gb": round(torch.cuda.memory_allocated(device) / 1e9, 3),
        "gpu_mem_reserved_gb": round(torch.cuda.memory_reserved(device) / 1e9, 3),
        "gpu_mem_peak_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 3),
    }


def ensure_out_dir(path: str | Path) -> Path:
    out_dir = Path(path)
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir


def write_json(path: Path, payload: Any) -> None:
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Wrote {path}")


def write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    if not rows:
        print(f"No rows to write for {path}")
        return

    fieldnames = sorted({key for row in rows for key in row.keys()})
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"Wrote {path}")


def filter_config_kwargs(config_class: type, raw_cfg: dict[str, Any]) -> dict[str, Any]:
    if is_dataclass(config_class):
        allowed = {f.name for f in fields(config_class)}
        return {k: v for k, v in raw_cfg.items() if k in allowed}
    return raw_cfg


def get_checkpoint_config_dict(ckpt: dict[str, Any]) -> dict[str, Any]:
    if isinstance(ckpt.get("config"), dict):
        return ckpt["config"]
    if isinstance(ckpt.get("model_config"), dict):
        return ckpt["model_config"]
    if isinstance(ckpt.get("extra"), dict):
        extra = ckpt["extra"]
        if isinstance(extra.get("config"), dict):
            return extra["config"]
    raise KeyError("Could not find config in checkpoint. Expected `config` or `model_config`.")


def get_state_dict_from_checkpoint(ckpt: dict[str, Any]) -> dict[str, torch.Tensor]:
    if "model" in ckpt:
        return ckpt["model"]
    if "state_dict" in ckpt:
        return ckpt["state_dict"]
    raise KeyError("Could not find model state dict. Expected `model` or `state_dict`.")


def load_one_model_from_notebook(
    arch: str,
    checkpoint_path: str | Path,
    device: torch.device,
    strict: bool = True,
):
    if arch not in ARCH_CLASSES:
        raise ValueError(f"Unknown arch: {arch}. Expected one of {list(ARCH_CLASSES)}")

    config_class = require_global_class(ARCH_CLASSES[arch]["config"])
    model_class = require_global_class(ARCH_CLASSES[arch]["model"])

    checkpoint_path = Path(checkpoint_path)
    ckpt = torch.load(checkpoint_path, map_location=device)

    raw_cfg = get_checkpoint_config_dict(ckpt)
    cfg_kwargs = filter_config_kwargs(config_class, raw_cfg)
    cfg = config_class(**cfg_kwargs)

    if hasattr(cfg, "validate"):
        cfg.validate()

    model = model_class(cfg).to(device)
    state = get_state_dict_from_checkpoint(ckpt)

    load_result = model.load_state_dict(state, strict=strict)

    if not strict:
        missing, unexpected = load_result
        if missing:
            print(f"[{arch}] missing keys: {missing[:10]}{'...' if len(missing) > 10 else ''}")
        if unexpected:
            print(f"[{arch}] unexpected keys: {unexpected[:10]}{'...' if len(unexpected) > 10 else ''}")

    model.eval()

    meta = {
        "arch": arch,
        "checkpoint_path": str(checkpoint_path),
        "config": cfg_kwargs,
        "n_params": model.num_parameters() if hasattr(model, "num_parameters") else None,
        "max_seq_len": getattr(cfg, "max_seq_len", None),
    }

    return model, cfg, meta

In [7]:
# Dataset utilities

class EvalPackedTokenDataset(Dataset):
    def __init__(self, token_ids: list[int], seq_len: int):
        if len(token_ids) < seq_len + 1:
            raise ValueError(
                f"Need at least seq_len + 1 tokens, got {len(token_ids)} "
                f"for seq_len={seq_len}."
            )
        self.data = torch.tensor(token_ids, dtype=torch.long)
        self.seq_len = seq_len
        self.n_samples = (len(self.data) - 1) // seq_len

    def __len__(self) -> int:
        return self.n_samples

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        start = idx * self.seq_len
        chunk = self.data[start : start + self.seq_len + 1]
        return chunk[:-1], chunk[1:]


def row_to_text(row: dict) -> str:
    for key in ("text", "content", "story"):
        value = row.get(key)
        if isinstance(value, str):
            return value
    return ""


def load_eval_tokens(
    dataset_name: str,
    tokenizer,
    split: str = "train",
    max_tokens: Optional[int] = None,
) -> list[int]:
    if dataset_name.startswith("file:"):
        path = dataset_name[5:]
        text = Path(path).read_text(encoding="utf-8")
        ids = tokenizer.encode(text)
        return ids[:max_tokens] if max_tokens else ids

    from datasets import load_dataset

    ds_configs = {
        "wikitext-2": ("wikitext", "wikitext-2-raw-v1", False),
        "wikitext-103": ("wikitext", "wikitext-103-raw-v1", False),
        "tinystories": ("roneneldan/TinyStories", None, False),
        "openwebtext": ("Skylion007/openwebtext", None, True),
        "fineweb-edu": ("HuggingFaceFW/fineweb-edu-score-2", "sample-10BT", True),
        "slimpajama": ("cerebras/SlimPajama-627B", None, True),
        "c4": ("allenai/c4", "en", True),
    }

    if dataset_name not in ds_configs:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    ds_name, ds_config, streaming = ds_configs[dataset_name]
    limit = max_tokens or 500_000_000
    tokens: list[int] = []

    if streaming:
        ds = load_dataset(ds_name, ds_config, split=split, streaming=True)
        for row in ds:
            text = row_to_text(row)
            if text.strip():
                tokens.extend(tokenizer.encode(text))
            if len(tokens) >= limit:
                tokens = tokens[:limit]
                break
    else:
        ds = load_dataset(ds_name, ds_config, split=split)
        for row in ds:
            text = row_to_text(row)
            if text.strip():
                tokens.extend(tokenizer.encode(text))
            if len(tokens) >= limit:
                tokens = tokens[:limit]
                break

    return tokens


def make_eval_data_splits(
    dataset_name: str,
    tokenizer,
    max_tokens: Optional[int],
    val_monitor_fraction: float,
    val_ablation_fraction: float,
    test_replication_fraction: float,
    val_monitor_max_tokens: Optional[int],
    val_ablation_max_tokens: Optional[int],
    test_replication_max_tokens: Optional[int],
) -> tuple[dict[str, list[int]], dict[str, Any]]:

    def cap(tokens: list[int], n: Optional[int]) -> list[int]:
        return tokens[:n] if n is not None else tokens

    info: dict[str, Any] = {
        "dataset": dataset_name,
        "fractions": {
            "val_monitor": val_monitor_fraction,
            "val_ablation": val_ablation_fraction,
            "test_replication": test_replication_fraction,
        },
    }

    if dataset_name in {"wikitext-2", "wikitext-103"}:
        train_all = load_eval_tokens(dataset_name, tokenizer, split="train", max_tokens=max_tokens)
        val_monitor = load_eval_tokens(dataset_name, tokenizer, split="validation", max_tokens=val_monitor_max_tokens)
        test_replication = load_eval_tokens(dataset_name, tokenizer, split="test", max_tokens=test_replication_max_tokens)

        n_ablation = max(int(len(train_all) * val_ablation_fraction), 1)
        if val_ablation_max_tokens is not None:
            n_ablation = min(n_ablation, val_ablation_max_tokens)

        train = train_all[:-n_ablation]
        val_ablation = train_all[-n_ablation:]

        info["split_policy"] = "official validation/test; val_ablation carved from train tail"

    else:
        all_tokens = load_eval_tokens(dataset_name, tokenizer, split="train", max_tokens=max_tokens)
        n_total = len(all_tokens)

        n_monitor = max(int(n_total * val_monitor_fraction), 1)
        n_ablation = max(int(n_total * val_ablation_fraction), 1)
        n_test = max(int(n_total * test_replication_fraction), 1)

        if val_monitor_max_tokens is not None:
            n_monitor = min(n_monitor, val_monitor_max_tokens)
        if val_ablation_max_tokens is not None:
            n_ablation = min(n_ablation, val_ablation_max_tokens)
        if test_replication_max_tokens is not None:
            n_test = min(n_test, test_replication_max_tokens)

        heldout_total = n_monitor + n_ablation + n_test
        if n_total - heldout_total < 2:
            raise ValueError("Training split too small after held-out split.")

        train_end = n_total - heldout_total
        monitor_end = train_end + n_monitor
        ablation_end = monitor_end + n_ablation

        train = all_tokens[:train_end]
        val_monitor = all_tokens[train_end:monitor_end]
        val_ablation = all_tokens[monitor_end:ablation_end]
        test_replication = all_tokens[ablation_end:]

        info["split_policy"] = "tail holdout from train token stream"
        info["n_total_loaded_tokens"] = n_total

    splits = {
        "train": train,
        "val_monitor": cap(val_monitor, val_monitor_max_tokens),
        "val_ablation": cap(val_ablation, val_ablation_max_tokens),
        "test_replication": cap(test_replication, test_replication_max_tokens),
    }

    info["n_tokens"] = {k: len(v) for k, v in splits.items()}
    return splits, info

In [8]:
# Forward, LM eval, generation speed

def unpack_logits_loss(output: Any) -> tuple[torch.Tensor, Optional[torch.Tensor]]:
    if not isinstance(output, tuple):
        raise TypeError("Model forward output must be a tuple.")
    logits = output[0]
    loss = output[1] if len(output) > 1 else None
    return logits, loss



@torch.no_grad()
def generate_ids_for_arch(
    model,
    arch: str,
    input_ids: torch.Tensor,
    max_new_tokens: int,
    temperature: float,
    top_k: int,
    mode: str,
) -> torch.Tensor:
    if arch == "transformer":
        if mode in {"auto", "cached", "recurrent"}:
            return model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_k=top_k,
                use_cache=True,
            )
        if mode == "full_prefix":
            return model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_k=top_k,
                use_cache=False,
            )
        raise ValueError(f"Unsupported Transformer generation mode: {mode}")

    if arch in {"mamba", "kimi"}:
        if mode in {"auto", "recurrent", "cached"}:
            return model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_k=top_k,
                use_recurrent=True,
                optimized_prefill=True,
            )
        if mode == "full_prefix":
            return model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_k=top_k,
                use_recurrent=False,
            )
        raise ValueError(f"Unsupported {arch} generation mode: {mode}")

    raise ValueError(f"Unknown arch: {arch}")


In [8]:
#===========================================================================

In [8]:
# HERE IS WHERE EVALUATION BEGINS

In [ ]:
#===========================================================================

In [9]:
# General utilities

def normalize_text(s: str) -> str:
    return " ".join(str(s).strip().lower().split())


def count_parameters(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def safe_config_to_dict(config: Any) -> dict[str, Any]:
    if config is None:
        return {}

    if is_dataclass(config):
        return asdict(config)

    if hasattr(config, "__dict__"):
        out = {}
        for k, v in vars(config).items():
            if isinstance(v, (str, int, float, bool, type(None))):
                out[k] = v
        return out

    return {}


def prompt_hash(prompt: str) -> str:
    return hashlib.md5(prompt.encode("utf-8")).hexdigest()


def get_tokenizer_name(tokenizer: Any) -> str:
    if hasattr(tokenizer, "name_or_path"):
        return str(tokenizer.name_or_path)
    return tokenizer.__class__.__name__


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_rows_csv(rows: list[dict[str, Any]], path: str | Path) -> None:
    path = Path(path)

    if not rows:
        pd.DataFrame().to_csv(path, index=False)
        return

    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)


def make_run_metadata(
    model: torch.nn.Module,
    tokenizer: Any,
    arch: str,
    model_name: str,
    config: Optional[Any] = None,
    train_tokens: Optional[int] = None,
    val_loss: Optional[float] = None,
    checkpoint_step: Optional[int] = None,
    max_seq_len: Optional[int] = None,
    extra: Optional[dict[str, Any]] = None,
) -> dict[str, Any]:

    cfg = safe_config_to_dict(config)

    metadata = {
        "run_time": time.strftime("%Y-%m-%d_%H-%M-%S"),
        "model_name": model_name,
        "arch": arch,
        "n_params": count_parameters(model),
        "tokenizer_name": get_tokenizer_name(tokenizer),
        "train_tokens": train_tokens,
        "val_loss": val_loss,
        "checkpoint_step": checkpoint_step,
        "max_seq_len": max_seq_len,
    }

    for key in [
        "d_model",
        "n_layers",
        "n_heads",
        "d_head",
        "d_state",
        "expand",
        "vocab_size",
        "dropout",
        "tie_embeddings",
    ]:
        if key in cfg:
            metadata[key] = cfg[key]

    if extra:
        metadata.update(extra)

    return metadata


def add_common_metadata(
    row: dict[str, Any],
    metadata: dict[str, Any],
) -> dict[str, Any]:
    out = dict(metadata)
    out.update(row)
    return out


In [ ]:
# Filler generation

FILLER_SENTENCES = [
    "Lily walked through the garden and looked at the small flowers.",
    "The sun was warm, and the birds sang in the trees.",
    "Tom found a little stone near the path and put it in his pocket.",
    "Anna smiled because the puppy was running around the yard.",
    "The children played quietly while their mother made lunch.",
    "A small bird sat on the fence and watched them.",
    "After a while, everyone went inside to drink some water.",
    "The room was bright, and the toys were on the floor.",
]

In [ ]:
def make_filler_to_token_budget(
    tokenizer,
    target_tokens: int,
    rng: random.Random,
) -> str:

    # Createw filler text with approximately target_tokens tokens
    # Important: filler sentences avoid the passkey candidate values

    if target_tokens <= 0:
        return ""

    text = ""

    while len(tokenizer.encode(text)) < target_tokens:
        sent = rng.choice(FILLER_SENTENCES)
        text += sent + " "

    ids = tokenizer.encode(text)[:target_tokens]
    return tokenizer.decode(ids)

In [ ]:
# Continuation log-probability scoring

@torch.no_grad()
def continuation_logprob(
    model,
    tokenizer,
    prompt: str,
    continuation: str,
    device: torch.device,
    amp_dtype: torch.dtype = torch.float16,
    use_amp: bool = True,
    max_seq_len: int = 1024,
) -> dict[str, Any]:
    #Computes log P(continuation | prompt)

    #Returns total and average logprob over continuation tokens

    prompt_ids = tokenizer.encode(prompt)
    cont_ids = tokenizer.encode(continuation)

    if len(cont_ids) == 0:
        raise ValueError(f"Continuation tokenized to empty sequence: {continuation!r}")

    full_ids = prompt_ids + cont_ids

    # Truncate from the left if needed, but never remove the continuation boundary.
    truncated = False
    removed_tokens = 0

    if len(full_ids) > max_seq_len:
        overflow = len(full_ids) - max_seq_len

        if overflow >= len(prompt_ids):
            raise ValueError(
                "Prompt too long: truncation would remove the answer boundary. "
                f"prompt_tokens={len(prompt_ids)}, cont_tokens={len(cont_ids)}, "
                f"max_seq_len={max_seq_len}"
            )

        prompt_ids = prompt_ids[overflow:]
        full_ids = prompt_ids + cont_ids
        truncated = True
        removed_tokens = overflow

    input_ids = torch.tensor(
        full_ids[:-1],
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)

    target_ids = torch.tensor(
        full_ids[1:],
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)

    prompt_len = len(prompt_ids)
    cont_len = len(cont_ids)

    # The first continuation token is predicted by the logit at position prompt_len - 1.
    start = prompt_len - 1
    end = start + cont_len

    model.eval()

    with torch.amp.autocast(
        device_type=device.type,
        dtype=amp_dtype,
        enabled=use_amp,
    ):
        output = model(input_ids)
        logits, _ = unpack_logits_loss(output)

    logits_cont = logits[:, start:end, :]
    targets_cont = target_ids[:, start:end]

    log_probs = F.log_softmax(logits_cont.float(), dim=-1)
    token_log_probs = log_probs.gather(
        -1,
        targets_cont.unsqueeze(-1),
    ).squeeze(-1)

    return {
        "total_logprob": float(token_log_probs.sum().item()),
        "avg_logprob": float(token_log_probs.mean().item()),
        "n_tokens": int(cont_len),
        "truncated": int(truncated),
        "removed_tokens": int(removed_tokens),
    }


def score_forced_choice(
    model,
    tokenizer,
    prompt: str,
    choices: dict[str, str],
    correct_key: str,
    device: torch.device,
    amp_dtype: torch.dtype,
    use_amp: bool,
    max_seq_len: int,
    score_by: str = "avg_logprob",
) -> dict[str, Any]:
    # Scores fixed candidate continuations.


    if score_by not in {"avg_logprob", "total_logprob"}:
        raise ValueError("score_by must be 'avg_logprob' or 'total_logprob'.")

    scores = {}

    for key, continuation in choices.items():
        scores[key] = continuation_logprob(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            continuation=continuation,
            device=device,
            amp_dtype=amp_dtype,
            use_amp=use_amp,
            max_seq_len=max_seq_len,
        )

    pred_key = max(scores.keys(), key=lambda k: scores[k][score_by])

    wrong_keys = [k for k in scores.keys() if k != correct_key]

    best_wrong_key = max(
        wrong_keys,
        key=lambda k: scores[k][score_by],
    )

    correct_score = scores[correct_key][score_by]
    best_wrong_score = scores[best_wrong_key][score_by]

    result = {
        "prediction": pred_key,
        "expected": correct_key,
        "is_correct": int(pred_key == correct_key),
        "score_by": score_by,
        "correct_score": float(correct_score),
        "best_wrong": best_wrong_key,
        "best_wrong_score": float(best_wrong_score),
        "margin": float(correct_score - best_wrong_score),
    }

    for key, score in scores.items():
        safe_key = str(key).replace(" ", "_")
        result[f"{safe_key}_avg_logprob"] = score["avg_logprob"]
        result[f"{safe_key}_total_logprob"] = score["total_logprob"]
        result[f"{safe_key}_n_tokens"] = score["n_tokens"]
        result[f"{safe_key}_truncated"] = score["truncated"]

    return result

In [ ]:
# Passkey retrieval tests

PASSKEY_KEYS = [
    "color",
    "animal",
    "place",
    "toy",
    "fruit",
    "shape",
    "name",
    "code",
]

# Synthetic values reduce accidental matches from TinyStories-style data.
PASSKEY_VALUES = [
    "dax",
    "miv",
    "zup",
    "keb",
    "norl",
    "vash",
    "toma",
    "lirp",
]

In [ ]:
def make_passkey_prompt(
    tokenizer,
    key: str,
    value: str,
    total_context_tokens: int,
    needle_position: float,
    rng: random.Random,
    style: str = "assignment",
) -> str:
    # Builds a passkey prompts
    #This is intentionally simple and less instruction-heavy than natural QA


    if style == "assignment":
        needle = f"{key} = {value}\n"
        question = f"\n{key} ="
    elif style == "qa":
        needle = f"Remember this fact: the secret {key} is {value}.\n"
        question = f"\nQuestion: What is the secret {key}?\nAnswer:"
    else:
        raise ValueError("style must be 'assignment' or 'qa'.")

    needle_tokens = len(tokenizer.encode(needle))
    question_tokens = len(tokenizer.encode(question))

    filler_budget = max(
        0,
        total_context_tokens - needle_tokens - question_tokens,
    )

    before_budget = int(filler_budget * needle_position)
    after_budget = filler_budget - before_budget

    before = make_filler_to_token_budget(tokenizer, before_budget, rng)
    after = make_filler_to_token_budget(tokenizer, after_budget, rng)

    prompt = before + "\n" + needle + "\n" + after + question

    return prompt

In [ ]:
def build_passkey_tests(
    tokenizer,
    max_seq_len: int,
    lengths: Optional[list[int]] = None,
    positions: Optional[list[float]] = None,
    seeds: Optional[list[int]] = None,
    style: str = "assignment",
) -> list[dict[str, Any]]:

    if lengths is None:
        lengths = [128, 256, 512, min(900, max_seq_len - 64)]

    lengths = sorted(set([x for x in lengths if x > 64 and x < max_seq_len]))

    if positions is None:
        positions = [0.05, 0.25, 0.50, 0.75, 0.95]

    if seeds is None:
        seeds = list(range(20))

    tests = []

    for seed in seeds:
        rng = random.Random(seed)

        for total_len in lengths:
            for pos in positions:
                key = rng.choice(PASSKEY_KEYS)
                value = rng.choice(PASSKEY_VALUES)

                prompt = make_passkey_prompt(
                    tokenizer=tokenizer,
                    key=key,
                    value=value,
                    total_context_tokens=total_len,
                    needle_position=pos,
                    rng=rng,
                    style=style,
                )

                choices = {
                    candidate: " " + candidate
                    for candidate in PASSKEY_VALUES
                }

                tests.append({
                    "test_type": "passkey",
                    "test_id": f"passkey_len{total_len}_pos{pos}_seed{seed}",
                    "prompt": prompt,
                    "choices": choices,
                    "expected": value,
                    "key": key,
                    "context_tokens": len(tokenizer.encode(prompt)),
                    "target_context_tokens": total_len,
                    "needle_position": pos,
                    "seed": seed,
                    "prompt_style": style,
                    "prompt_hash": prompt_hash(prompt),
                })

    return tests

In [ ]:
def build_passkey_distractor_tests(
    tokenizer,
    max_seq_len: int,
    lengths: Optional[list[int]] = None,
    positions: Optional[list[float]] = None,
    seeds: Optional[list[int]] = None,
    n_facts: int = 4,
) -> list[dict[str, Any]]:
    # Multi-fact associative recall.
    # Tests whether the model retrieves the value for the requested key.

    if lengths is None:
        lengths = [128, 256, 512, min(900, max_seq_len - 64)]

    lengths = sorted(set([x for x in lengths if x > 64 and x < max_seq_len]))

    if positions is None:
        positions = [0.05, 0.25, 0.50, 0.75, 0.95]

    if seeds is None:
        seeds = list(range(20))

    tests = []

    for seed in seeds:
        rng = random.Random(10_000 + seed)

        for total_len in lengths:
            for pos in positions:
                keys = rng.sample(PASSKEY_KEYS, k=min(n_facts, len(PASSKEY_KEYS)))
                values = rng.sample(PASSKEY_VALUES, k=len(keys))

                facts = list(zip(keys, values))
                query_key, expected_value = rng.choice(facts)

                fact_block = "".join([f"{k} = {v}\n" for k, v in facts])
                question = f"\n{query_key} ="

                fact_tokens = len(tokenizer.encode(fact_block))
                question_tokens = len(tokenizer.encode(question))

                filler_budget = max(
                    0,
                    total_len - fact_tokens - question_tokens,
                )

                before_budget = int(filler_budget * pos)
                after_budget = filler_budget - before_budget

                before = make_filler_to_token_budget(tokenizer, before_budget, rng)
                after = make_filler_to_token_budget(tokenizer, after_budget, rng)

                prompt = before + "\n" + fact_block + "\n" + after + question

                choices = {
                    candidate: " " + candidate
                    for candidate in PASSKEY_VALUES
                }

                tests.append({
                    "test_type": "passkey_distractor",
                    "test_id": f"passkey_distractor_len{total_len}_pos{pos}_seed{seed}",
                    "prompt": prompt,
                    "choices": choices,
                    "expected": expected_value,
                    "key": query_key,
                    "context_tokens": len(tokenizer.encode(prompt)),
                    "target_context_tokens": total_len,
                    "needle_position": pos,
                    "seed": seed,
                    "n_facts": len(facts),
                    "facts_json": json.dumps(dict(facts)),
                    "prompt_style": "assignment_distractor",
                    "prompt_hash": prompt_hash(prompt),
                })

    return tests

In [ ]:
def build_passkey_update_tests(
    tokenizer,
    max_seq_len: int,
    lengths: Optional[list[int]] = None,
    seeds: Optional[list[int]] = None,
) -> list[dict[str, Any]]:
    # Correct answer is the latest value

    if lengths is None:
        lengths = [128, 256, 512, min(900, max_seq_len - 64)]

    lengths = sorted(set([x for x in lengths if x > 64 and x < max_seq_len]))

    if seeds is None:
        seeds = list(range(20))

    tests = []

    for seed in seeds:
        rng = random.Random(20_000 + seed)

        for total_len in lengths:
            key = rng.choice(PASSKEY_KEYS)
            old_value, new_value = rng.sample(PASSKEY_VALUES, k=2)

            first_fact = f"{key} = {old_value}\n"
            second_fact = f"{key} = {new_value}\n"
            question = f"\n{key} ="

            fixed_tokens = (
                len(tokenizer.encode(first_fact))
                + len(tokenizer.encode(second_fact))
                + len(tokenizer.encode(question))
            )

            filler_budget = max(0, total_len - fixed_tokens)

            # Split filler into three regions:
            # before old fact, between facts, after new fact.
            before_budget = int(filler_budget * 0.20)
            middle_budget = int(filler_budget * 0.40)
            after_budget = filler_budget - before_budget - middle_budget

            before = make_filler_to_token_budget(tokenizer, before_budget, rng)
            middle = make_filler_to_token_budget(tokenizer, middle_budget, rng)
            after = make_filler_to_token_budget(tokenizer, after_budget, rng)

            prompt = (
                before + "\n"
                + first_fact + "\n"
                + middle + "\n"
                + second_fact + "\n"
                + after
                + question
            )

            choices = {
                candidate: " " + candidate
                for candidate in PASSKEY_VALUES
            }

            tests.append({
                "test_type": "passkey_update",
                "test_id": f"passkey_update_len{total_len}_seed{seed}",
                "prompt": prompt,
                "choices": choices,
                "expected": new_value,
                "old_value": old_value,
                "new_value": new_value,
                "key": key,
                "context_tokens": len(tokenizer.encode(prompt)),
                "target_context_tokens": total_len,
                "seed": seed,
                "prompt_style": "assignment_update",
                "prompt_hash": prompt_hash(prompt),
            })

    return tests

In [ ]:
# Entity and simple reasoning tests

NAMES = [
    "Lily", "Tom", "Anna", "Ben", "Emma", "Jack", "Mia", "Sara",
    "Mike", "Nora", "Leo", "Eva", "Omar", "Tina", "Sam", "Ruby",
]

OBJECTS = [
    "ball", "kite", "cup", "dog", "shell", "book", "spoon", "box",
    "hat", "key", "apple", "stone", "toy", "bag", "coin", "pencil",
]

COLORS = [
    "red", "blue", "yellow", "green", "white", "black", "purple", "orange",
]

In [ ]:
def build_entity_binding_tests(
    seeds: Optional[list[int]] = None,
) -> list[dict[str, Any]]:
    """
    Forced-choice entity/attribute/ownership tests.

    These are templated and randomized to create many examples.
    """

    if seeds is None:
        seeds = list(range(100))

    tests = []

    for seed in seeds:
        rng = random.Random(30_000 + seed)

        # Attribute binding: What color was X's object?
        name1, name2 = rng.sample(NAMES, k=2)
        object1, object2 = rng.sample(OBJECTS, k=2)
        color1, color2 = rng.sample(COLORS, k=2)

        prompt = (
            f"{name1} had a {color1} {object1}. "
            f"{name2} had a {color2} {object2}.\n"
            f"Question: What color was {name1}'s {object1}?\n"
            f"Answer:"
        )

        choices = {c: " " + c for c in COLORS}

        tests.append({
            "test_type": "entity_attribute",
            "test_id": f"entity_attribute_seed{seed}",
            "prompt": prompt,
            "choices": choices,
            "expected": color1,
            "seed": seed,
            "prompt_hash": prompt_hash(prompt),
        })

        # Ownership: What did X have?
        name1, name2 = rng.sample(NAMES, k=2)
        object1, object2 = rng.sample(OBJECTS, k=2)
        color1, color2 = rng.sample(COLORS, k=2)

        prompt = (
            f"{name1} had a {color1} {object1}. "
            f"{name2} had a {color2} {object2}. "
            f"The children played in the yard.\n"
            f"Question: What did {name2} have?\n"
            f"Answer:"
        )

        candidate_objects = rng.sample(
            [x for x in OBJECTS if x not in {object1, object2}],
            k=6,
        ) + [object1, object2]

        choices = {obj: " " + obj for obj in candidate_objects}

        tests.append({
            "test_type": "entity_owner",
            "test_id": f"entity_owner_seed{seed}",
            "prompt": prompt,
            "choices": choices,
            "expected": object2,
            "seed": seed,
            "prompt_hash": prompt_hash(prompt),
        })

        # Transfer: Who has the object at the end?
        giver, receiver, distractor = rng.sample(NAMES, k=3)
        obj = rng.choice(OBJECTS)

        prompt = (
            f"{giver} found a {obj}. "
            f"{giver} gave the {obj} to {receiver}. "
            f"{receiver} put it in a bag.\n"
            f"Question: Who had the {obj} at the end?\n"
            f"Answer:"
        )

        choices = {
            name: " " + name
            for name in [giver, receiver, distractor]
        }

        tests.append({
            "test_type": "entity_transfer",
            "test_id": f"entity_transfer_seed{seed}",
            "prompt": prompt,
            "choices": choices,
            "expected": receiver,
            "seed": seed,
            "prompt_hash": prompt_hash(prompt),
        })

        # Update: final attribute after change
        name = rng.choice(NAMES)
        obj = rng.choice(OBJECTS)
        old_color, new_color = rng.sample(COLORS, k=2)

        prompt = (
            f"{name} had a {old_color} {obj}. "
            f"Later, {name} painted the {obj} {new_color}.\n"
            f"Question: What color was the {obj} at the end?\n"
            f"Answer:"
        )

        choices = {c: " " + c for c in COLORS}

        tests.append({
            "test_type": "entity_update",
            "test_id": f"entity_update_seed{seed}",
            "prompt": prompt,
            "choices": choices,
            "expected": new_color,
            "old_color": old_color,
            "new_color": new_color,
            "seed": seed,
            "prompt_hash": prompt_hash(prompt),
        })

    return tests

In [ ]:
# Multiple-choice tests with shuffled labels

BASE_MC_TESTS = [
    {
        "test_type": "mc_reasoning",
        "id": "reason_broken_glass",
        "story": (
            "Emma dropped the glass. It broke on the floor. "
            "Her mother brought a broom."
        ),
        "question": "Why did her mother bring a broom?",
        "correct_answer": "to clean the broken glass",
        "wrong_answers": [
            "to eat dinner",
            "to play outside",
            "to sleep",
        ],
    },
    {
        "test_type": "mc_reasoning",
        "id": "reason_raincoat",
        "story": (
            "Tom looked outside and saw rain. "
            "He put on his raincoat before school."
        ),
        "question": "Why did Tom put on his raincoat?",
        "correct_answer": "because it was raining",
        "wrong_answers": [
            "because he was hungry",
            "because he wanted to sleep",
            "because the sun was too hot",
        ],
    },
    {
        "test_type": "mc_entity",
        "id": "mc_entity_owner",
        "story": (
            "Sara had a red balloon. Mike had a blue car. "
            "Sara tied the balloon to a chair."
        ),
        "question": "What did Sara have?",
        "correct_answer": "a red balloon",
        "wrong_answers": [
            "a blue car",
            "a brown dog",
            "a yellow cup",
        ],
    },
    {
        "test_type": "mc_transfer",
        "id": "mc_transfer_shell",
        "story": (
            "Emma found a shell. She gave the shell to Jack. "
            "Jack put it in his bag."
        ),
        "question": "Who had the shell at the end?",
        "correct_answer": "Jack",
        "wrong_answers": [
            "Emma",
            "Lily",
            "Anna",
        ],
    },
    {
        "test_type": "mc_update",
        "id": "mc_update_painted_cup",
        "story": (
            "Anna had a red cup. Later, Anna painted the cup blue. "
            "The cup dried near the window."
        ),
        "question": "What color was the cup at the end?",
        "correct_answer": "blue",
        "wrong_answers": [
            "red",
            "green",
            "yellow",
        ],
    },
]

In [ ]:
def build_mc_tests_shuffled(
    seeds: Optional[list[int]] = None,
) -> list[dict[str, Any]]:
    # Each seed reshuffles answer positions, avoiding the all-correct-A problem.

    if seeds is None:
        seeds = list(range(100))

    tests = []
    labels = ["A", "B", "C", "D"]

    for seed in seeds:
        rng = random.Random(40_000 + seed)

        for item in BASE_MC_TESTS:
            answers = [item["correct_answer"]] + list(item["wrong_answers"])
            rng.shuffle(answers)

            label_to_answer = dict(zip(labels, answers))

            correct_label = None
            for label, answer in label_to_answer.items():
                if answer == item["correct_answer"]:
                    correct_label = label
                    break

            assert correct_label is not None

            prompt = (
                f"{item['story']}\n\n"
                f"Question: {item['question']}\n"
                f"A. {label_to_answer['A']}\n"
                f"B. {label_to_answer['B']}\n"
                f"C. {label_to_answer['C']}\n"
                f"D. {label_to_answer['D']}\n"
                f"Answer:"
            )

            tests.append({
                "test_type": item["test_type"],
                "test_id": f"{item['id']}_seed{seed}",
                "base_id": item["id"],
                "prompt": prompt,

                # Letter scoring: evaluates whether the model outputs A/B/C/D
                "choices_letter": {
                    "A": " A",
                    "B": " B",
                    "C": " C",
                    "D": " D",
                },

                # Text scoring: evaluates whether the model prefers answer content
                "choices_text": {
                    label: " " + answer
                    for label, answer in label_to_answer.items()
                },

                "correct": correct_label,
                "correct_answer_text": item["correct_answer"],
                "label_to_answer_json": json.dumps(label_to_answer),
                "seed": seed,
                "prompt_hash": prompt_hash(prompt),
            })

    return tests

In [ ]:
# Running forced-choice test sets

def run_forced_choice_test_set(
    model,
    tokenizer,
    tests: list[dict[str, Any]],
    device: torch.device,
    amp_dtype: torch.dtype,
    use_amp: bool,
    max_seq_len: int,
    metadata: dict[str, Any],
    score_by: str = "avg_logprob",
) -> list[dict[str, Any]]:

    rows = []

    for i, test in enumerate(tests):
        result = score_forced_choice(
            model=model,
            tokenizer=tokenizer,
            prompt=test["prompt"],
            choices=test["choices"],
            correct_key=test["expected"],
            device=device,
            amp_dtype=amp_dtype,
            use_amp=use_amp,
            max_seq_len=max_seq_len,
            score_by=score_by,
        )

        row = {
            "eval_family": "forced_choice",
            "test_type": test["test_type"],
            "test_id": test["test_id"],
            "prompt_hash": test.get("prompt_hash"),
            "seed": test.get("seed"),
            "expected": test["expected"],
            "prediction": result["prediction"],
            "is_correct": result["is_correct"],
            "score_by": result["score_by"],
            "correct_score": result["correct_score"],
            "best_wrong": result["best_wrong"],
            "best_wrong_score": result["best_wrong_score"],
            "margin": result["margin"],
            "context_tokens": test.get("context_tokens"),
            "target_context_tokens": test.get("target_context_tokens"),
            "needle_position": test.get("needle_position"),
            "key": test.get("key"),
            "prompt_style": test.get("prompt_style"),
            "n_facts": test.get("n_facts"),
            "facts_json": test.get("facts_json"),
            "old_value": test.get("old_value"),
            "new_value": test.get("new_value"),
        }

        # Add per-choice logprobs.
        for k, v in result.items():
            if k not in row:
                row[k] = v

        rows.append(add_common_metadata(row, metadata))

    return rows


def run_mc_test_set(
    model,
    tokenizer,
    tests: list[dict[str, Any]],
    device: torch.device,
    amp_dtype: torch.dtype,
    use_amp: bool,
    max_seq_len: int,
    metadata: dict[str, Any],
    score_by: str = "avg_logprob",
) -> list[dict[str, Any]]:

    rows = []

    for test in tests:
        # 1. Letter scoring
        letter_result = score_forced_choice(
            model=model,
            tokenizer=tokenizer,
            prompt=test["prompt"],
            choices=test["choices_letter"],
            correct_key=test["correct"],
            device=device,
            amp_dtype=amp_dtype,
            use_amp=use_amp,
            max_seq_len=max_seq_len,
            score_by=score_by,
        )

        row = {
            "eval_family": "mc",
            "mc_scoring_mode": "letter",
            "test_type": test["test_type"],
            "test_id": test["test_id"],
            "base_id": test["base_id"],
            "seed": test["seed"],
            "prompt_hash": test["prompt_hash"],
            "expected": test["correct"],
            "prediction": letter_result["prediction"],
            "is_correct": letter_result["is_correct"],
            "score_by": letter_result["score_by"],
            "correct_score": letter_result["correct_score"],
            "best_wrong": letter_result["best_wrong"],
            "best_wrong_score": letter_result["best_wrong_score"],
            "margin": letter_result["margin"],
            "correct_answer_text": test["correct_answer_text"],
            "label_to_answer_json": test["label_to_answer_json"],
        }

        for k, v in letter_result.items():
            if k not in row:
                row[k] = v

        rows.append(add_common_metadata(row, metadata))

        # 2. Text scoring
        text_result = score_forced_choice(
            model=model,
            tokenizer=tokenizer,
            prompt=test["prompt"],
            choices=test["choices_text"],
            correct_key=test["correct"],
            device=device,
            amp_dtype=amp_dtype,
            use_amp=use_amp,
            max_seq_len=max_seq_len,
            score_by=score_by,
        )

        row = {
            "eval_family": "mc",
            "mc_scoring_mode": "text",
            "test_type": test["test_type"],
            "test_id": test["test_id"],
            "base_id": test["base_id"],
            "seed": test["seed"],
            "prompt_hash": test["prompt_hash"],
            "expected": test["correct"],
            "prediction": text_result["prediction"],
            "is_correct": text_result["is_correct"],
            "score_by": text_result["score_by"],
            "correct_score": text_result["correct_score"],
            "best_wrong": text_result["best_wrong"],
            "best_wrong_score": text_result["best_wrong_score"],
            "margin": text_result["margin"],
            "correct_answer_text": test["correct_answer_text"],
            "label_to_answer_json": test["label_to_answer_json"],
        }

        for k, v in text_result.items():
            if k not in row:
                row[k] = v

        rows.append(add_common_metadata(row, metadata))

    return rows

In [ ]:
# Optional qualitative generation probes

@torch.no_grad()
def generate_text(
    model,
    arch: str,
    tokenizer,
    prompt: str,
    device: torch.device,
    max_seq_len: int,
    max_new_tokens: int = 16,
    temperature: float = 0.1,
    top_k: int = 1,
    mode: str = "auto",
) -> str:

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    input_ids = input_ids[:, -max_seq_len:]

    out = generate_ids_for_arch(
        model=model,
        arch=arch,
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        mode=mode,
    )

    new_ids = out[0, input_ids.size(1):].tolist()
    return tokenizer.decode(new_ids)


def run_qualitative_generation_examples(
    model,
    arch: str,
    tokenizer,
    tests: list[dict[str, Any]],
    device: torch.device,
    max_seq_len: int,
    metadata: dict[str, Any],
    n_examples: int = 20,
    max_new_tokens: int = 16,
    temperature: float = 0.1,
    top_k: int = 1,
    mode: str = "auto",
    seed: int = 123,
) -> list[dict[str, Any]]:

    rng = random.Random(seed)

    if len(tests) > n_examples:
        selected = rng.sample(tests, k=n_examples)
    else:
        selected = tests

    rows = []

    for test in selected:
        generated = generate_text(
            model=model,
            arch=arch,
            tokenizer=tokenizer,
            prompt=test["prompt"],
            device=device,
            max_seq_len=max_seq_len,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            mode=mode,
        )

        gen_norm = normalize_text(generated)
        expected = test.get("expected", "")
        exp_norm = normalize_text(expected)

        first_token = gen_norm.split()[0] if gen_norm.split() else ""
        first_token = first_token.strip(".,!?;:\"'()[]{}")

        row = {
            "eval_family": "qualitative_generation",
            "test_type": test.get("test_type"),
            "test_id": test.get("test_id"),
            "prompt_hash": test.get("prompt_hash"),
            "seed": test.get("seed"),
            "expected": expected,
            "generated": generated.strip(),
            "contains_expected": int(exp_norm in gen_norm) if exp_norm else None,
            "first_token_match": int(first_token == exp_norm) if exp_norm else None,
            "max_new_tokens": max_new_tokens,
            "temperature": temperature,
            "top_k": top_k,
            "generation_mode": mode,
            "context_tokens": test.get("context_tokens"),
            "target_context_tokens": test.get("target_context_tokens"),
            "needle_position": test.get("needle_position"),
        }

        rows.append(add_common_metadata(row, metadata))

    return rows

In [ ]:
# Efficiency evaluation

@torch.no_grad()
def measure_inference_efficiency(
    model,
    tokenizer,
    device: torch.device,
    metadata: dict[str, Any],
    context_lengths: Optional[list[int]] = None,
    batch_size: int = 1,
    n_warmup: int = 3,
    n_steps: int = 10,
    amp_dtype: torch.dtype = torch.float16,
    use_amp: bool = True,
) -> list[dict[str, Any]]:
    # Simple forward-pass efficiency test. Measures approximate forward tokens/sec and memory.

    if context_lengths is None:
        context_lengths = [128, 256, 512, 768, 1024]

    vocab_size = getattr(model, "vocab_size", None)
    if vocab_size is None:
        vocab_size = len(tokenizer)

    rows = []

    model.eval()

    for seq_len in context_lengths:
        if metadata.get("max_seq_len") is not None and seq_len > metadata["max_seq_len"]:
            continue

        input_ids = torch.randint(
            low=0,
            high=vocab_size,
            size=(batch_size, seq_len),
            device=device,
            dtype=torch.long,
        )

        if device.type == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        # Warmup
        for _ in range(n_warmup):
            with torch.amp.autocast(
                device_type=device.type,
                dtype=amp_dtype,
                enabled=use_amp,
            ):
                output = model(input_ids)
                logits, _ = unpack_logits_loss(output)

        if device.type == "cuda":
            torch.cuda.synchronize()

        start = time.time()

        for _ in range(n_steps):
            with torch.amp.autocast(
                device_type=device.type,
                dtype=amp_dtype,
                enabled=use_amp,
            ):
                output = model(input_ids)
                logits, _ = unpack_logits_loss(output)

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.time() - start

        total_tokens = batch_size * seq_len * n_steps
        tokens_per_sec = total_tokens / max(elapsed, 1e-9)
        ms_per_forward = 1000.0 * elapsed / n_steps

        peak_mem_mb = None
        if device.type == "cuda":
            peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

        row = {
            "eval_family": "efficiency",
            "seq_len": seq_len,
            "batch_size": batch_size,
            "n_steps": n_steps,
            "elapsed_sec": elapsed,
            "ms_per_forward": ms_per_forward,
            "tokens_per_sec": tokens_per_sec,
            "peak_mem_mb": peak_mem_mb,
        }

        rows.append(add_common_metadata(row, metadata))

    return rows

In [ ]:
# Summary helpers

def summarize_accuracy(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    if len(df) == 0:
        return pd.DataFrame()

    return (
        df
        .groupby(group_cols, dropna=False)
        .agg(
            n=("is_correct", "count"),
            accuracy=("is_correct", "mean"),
            mean_margin=("margin", "mean"),
            median_margin=("margin", "median"),
            mean_correct_score=("correct_score", "mean"),
            mean_best_wrong_score=("best_wrong_score", "mean"),
        )
        .reset_index()
        .sort_values(group_cols)
    )


def make_probe_summary_tables(
    forced_choice_rows: list[dict[str, Any]],
    mc_rows: list[dict[str, Any]],
    out_dir: str | Path,
) -> dict[str, pd.DataFrame]:

    out_dir = ensure_dir(out_dir)

    tables = {}

    fc_df = pd.DataFrame(forced_choice_rows)
    mc_df = pd.DataFrame(mc_rows)

    if len(fc_df) > 0:
        tables["forced_choice_by_test_type"] = summarize_accuracy(
            fc_df,
            ["model_name", "arch", "test_type"],
        )

        tables["passkey_by_length"] = summarize_accuracy(
            fc_df[fc_df["test_type"].astype(str).str.contains("passkey", na=False)],
            ["model_name", "arch", "test_type", "target_context_tokens"],
        )

        tables["passkey_by_position"] = summarize_accuracy(
            fc_df[
                fc_df["test_type"].astype(str).str.contains("passkey", na=False)
                & fc_df["needle_position"].notna()
            ],
            ["model_name", "arch", "test_type", "needle_position"],
        )

    if len(mc_df) > 0:
        tables["mc_by_scoring_mode"] = summarize_accuracy(
            mc_df,
            ["model_name", "arch", "mc_scoring_mode", "test_type"],
        )

    for name, table in tables.items():
        table.to_csv(out_dir / f"{name}.csv", index=False)

    return tables


In [ ]:
# Main evaluation runner

def run_full_probe_evaluation(
    model,
    arch: str,
    tokenizer,
    device: torch.device,
    max_seq_len: int,
    model_name: str,
    out_dir: str | Path = "probe_outputs",

    # Optional metadata
    config: Optional[Any] = None,
    train_tokens: Optional[int] = None,
    val_loss: Optional[float] = None,
    checkpoint_step: Optional[int] = None,
    extra_metadata: Optional[dict[str, Any]] = None,

    # Eval settings
    amp_dtype: torch.dtype = torch.float16,
    use_amp: bool = True,
    score_by: str = "avg_logprob",

    # Test sizes
    passkey_seeds: Optional[list[int]] = None,
    entity_seeds: Optional[list[int]] = None,
    mc_seeds: Optional[list[int]] = None,
    lengths: Optional[list[int]] = None,
    positions: Optional[list[float]] = None,

    # Optional qualitative generation
    run_generation_examples: bool = True,
    n_generation_examples: int = 20,

    # Optional efficiency
    run_efficiency: bool = True,
    efficiency_context_lengths: Optional[list[int]] = None,
) -> dict[str, Any]:

    out_dir = ensure_dir(out_dir)

    metadata = make_run_metadata(
        model=model,
        tokenizer=tokenizer,
        arch=arch,
        model_name=model_name,
        config=config,
        train_tokens=train_tokens,
        val_loss=val_loss,
        checkpoint_step=checkpoint_step,
        max_seq_len=max_seq_len,
        extra=extra_metadata,
    )

    run_id = (
        f"{metadata['run_time']}_"
        f"{model_name}_"
        f"{arch}_"
        f"{metadata['n_params']}"
    )

    metadata["run_id"] = run_id

    print(f"Running probe evaluation for {model_name} / {arch}")
    print(f"Run ID: {run_id}")
    print(f"Output directory: {out_dir}")

    with open(out_dir / f"{run_id}_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    # Build tests

    if passkey_seeds is None:
        passkey_seeds = list(range(20))

    if entity_seeds is None:
        entity_seeds = list(range(100))

    if mc_seeds is None:
        mc_seeds = list(range(100))

    if lengths is None:
        lengths = [128, 256, 512, min(900, max_seq_len - 64)]

    if positions is None:
        positions = [0.05, 0.25, 0.50, 0.75, 0.95]

    passkey_tests = build_passkey_tests(
        tokenizer=tokenizer,
        max_seq_len=max_seq_len,
        lengths=lengths,
        positions=positions,
        seeds=passkey_seeds,
        style="assignment",
    )

    passkey_distractor_tests = build_passkey_distractor_tests(
        tokenizer=tokenizer,
        max_seq_len=max_seq_len,
        lengths=lengths,
        positions=positions,
        seeds=passkey_seeds,
        n_facts=4,
    )

    passkey_update_tests = build_passkey_update_tests(
        tokenizer=tokenizer,
        max_seq_len=max_seq_len,
        lengths=lengths,
        seeds=passkey_seeds,
    )

    entity_tests = build_entity_binding_tests(
        seeds=entity_seeds,
    )

    mc_tests = build_mc_tests_shuffled(
        seeds=mc_seeds,
    )

    print(f"Passkey tests: {len(passkey_tests)}")
    print(f"Passkey distractor tests: {len(passkey_distractor_tests)}")
    print(f"Passkey update tests: {len(passkey_update_tests)}")
    print(f"Entity tests: {len(entity_tests)}")
    print(f"MC tests: {len(mc_tests)} base prompts, scored twice letter/text")

    # Run forced-choice tests

    all_forced_choice_tests = (
        passkey_tests
        + passkey_distractor_tests
        + passkey_update_tests
        + entity_tests
    )

    forced_choice_rows = run_forced_choice_test_set(
        model=model,
        tokenizer=tokenizer,
        tests=all_forced_choice_tests,
        device=device,
        amp_dtype=amp_dtype,
        use_amp=use_amp,
        max_seq_len=max_seq_len,
        metadata=metadata,
        score_by=score_by,
    )

    forced_choice_path = out_dir / f"{run_id}_forced_choice_results.csv"
    save_rows_csv(forced_choice_rows, forced_choice_path)
    print(f"Saved forced-choice results: {forced_choice_path}")

    # Run MC tests

    mc_rows = run_mc_test_set(
        model=model,
        tokenizer=tokenizer,
        tests=mc_tests,
        device=device,
        amp_dtype=amp_dtype,
        use_amp=use_amp,
        max_seq_len=max_seq_len,
        metadata=metadata,
        score_by=score_by,
    )

    mc_path = out_dir / f"{run_id}_mc_results.csv"
    save_rows_csv(mc_rows, mc_path)
    print(f"Saved MC results: {mc_path}")

    # Optional qualitative generation examples

    generation_rows = []

    if run_generation_examples:
        qualitative_pool = (
            passkey_tests[:]
            + passkey_distractor_tests[:]
            + passkey_update_tests[:]
            + entity_tests[:]
        )

        generation_rows = run_qualitative_generation_examples(
            model=model,
            arch=arch,
            tokenizer=tokenizer,
            tests=qualitative_pool,
            device=device,
            max_seq_len=max_seq_len,
            metadata=metadata,
            n_examples=n_generation_examples,
            max_new_tokens=16,
            temperature=0.1,
            top_k=1,
            mode="auto",
        )

        generation_path = out_dir / f"{run_id}_qualitative_generation_results.csv"
        save_rows_csv(generation_rows, generation_path)
        print(f"Saved qualitative generation examples: {generation_path}")

    # Optional efficiency evaluation

    efficiency_rows = []

    if run_efficiency:
        efficiency_rows = measure_inference_efficiency(
            model=model,
            tokenizer=tokenizer,
            device=device,
            metadata=metadata,
            context_lengths=efficiency_context_lengths or [128, 256, 512, 768, min(1024, max_seq_len)],
            batch_size=1,
            n_warmup=3,
            n_steps=10,
            amp_dtype=amp_dtype,
            use_amp=use_amp,
        )

        efficiency_path = out_dir / f"{run_id}_efficiency_results.csv"
        save_rows_csv(efficiency_rows, efficiency_path)
        print(f"Saved efficiency results: {efficiency_path}")

    # Save summary tables

    summary_tables = make_probe_summary_tables(
        forced_choice_rows=forced_choice_rows,
        mc_rows=mc_rows,
        out_dir=out_dir / f"{run_id}_summaries",
    )

    print("Saved summary tables.")

    return {
        "run_id": run_id,
        "metadata": metadata,
        "forced_choice_rows": forced_choice_rows,
        "mc_rows": mc_rows,
        "generation_rows": generation_rows,
        "efficiency_rows": efficiency_rows,
        "summary_tables": summary_tables,
        "paths": {
            "forced_choice": str(forced_choice_path),
            "mc": str(mc_path),
            "summaries": str(out_dir / f"{run_id}_summaries"),
        },
    }

In [10]:



def clear_model_from_memory(model=None, device: Optional[torch.device] = None):
    # Deletes current model and clears CUDA cache before the next architecture.

    if model is not None:
        del model

    gc.collect()

    if device is not None and device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()


def get_arch_checkpoint_map_from_cfg(cfg: ColabEvalConfig) -> dict[str, Optional[str]]:
    return {
        "transformer": cfg.TRANSFORMER_CKPT,
        "mamba": cfg.MAMBA_CKPT,
        "kimi": cfg.KIMI_CKPT,
    }


def make_model_name_from_checkpoint(arch: str, checkpoint_path: str | Path) -> str:

    checkpoint_path = Path(checkpoint_path)

    if checkpoint_path.name.endswith(".pt"):
        run_folder = checkpoint_path.parent.name
    else:
        run_folder = checkpoint_path.name

    return f"{arch}_{run_folder}"


In [ ]:
def run_all_models_probe_evaluation(
    cfg: ColabEvalConfig,
    tokenizer,
    out_dir: Optional[str | Path] = None,

    # Test-size controls
    quick_debug: bool = False,
    passkey_seeds: Optional[list[int]] = None,
    entity_seeds: Optional[list[int]] = None,
    mc_seeds: Optional[list[int]] = None,
    lengths: Optional[list[int]] = None,
    positions: Optional[list[float]] = None,

    # Evaluation flags
    run_generation_examples: bool = True,
    n_generation_examples: int = 20,
    run_efficiency: bool = True,
    efficiency_context_lengths: Optional[list[int]] = None,

    # Loading/eval behavior
    strict_load: Optional[bool] = None,
    score_by: str = "avg_logprob",
) -> dict[str, Any]:
    # Sequentially loads Transformer, Mamba, and Kimi checkpoints from CFG, runs the improved probe suite for each, and saves combined CSVs.
    # This intentionally does NOT keep all models in GPU memory at once

    device = choose_device(cfg.DEVICE)
    amp_dtype, use_amp = choose_amp_dtype(device)

    if out_dir is None:
        out_dir = Path(cfg.OUT_DIR) / "improved_probe_outputs"
    else:
        out_dir = Path(out_dir)

    out_dir.mkdir(parents=True, exist_ok=True)

    if strict_load is None:
        strict_load = cfg.STRICT_LOAD

    # Small settings for smoke test.
    if quick_debug:
        passkey_seeds = list(range(2)) if passkey_seeds is None else passkey_seeds
        entity_seeds = list(range(5)) if entity_seeds is None else entity_seeds
        mc_seeds = list(range(5)) if mc_seeds is None else mc_seeds
        n_generation_examples = min(n_generation_examples, 10)
    else:
        passkey_seeds = list(range(20)) if passkey_seeds is None else passkey_seeds
        entity_seeds = list(range(100)) if entity_seeds is None else entity_seeds
        mc_seeds = list(range(100)) if mc_seeds is None else mc_seeds

    arch_to_ckpt = get_arch_checkpoint_map_from_cfg(cfg)

    all_results = {}
    all_forced_rows = []
    all_mc_rows = []
    all_generation_rows = []
    all_efficiency_rows = []
    all_metadata = []

    print("=" * 80)
    print("Running probe evaluation for all configured models")
    print(f"Device: {device}")
    print(f"AMP dtype: {amp_dtype}, use_amp={use_amp}")
    print(f"Output dir: {out_dir}")
    print("=" * 80)

    for arch, ckpt_path in arch_to_ckpt.items():
        if ckpt_path is None:
            print(f"\nSkipping {arch}: checkpoint path is None")
            continue

        ckpt_path = Path(ckpt_path)

        if not ckpt_path.exists():
            print(f"\nSkipping {arch}: checkpoint not found at {ckpt_path}")
            continue

        print("\n" + "=" * 80)
        print(f"Loading {arch}")
        print(f"Checkpoint: {ckpt_path}")
        print("=" * 80)

        clear_model_from_memory(device=device)

        model, model_cfg, load_meta = load_one_model_from_notebook(
            arch=arch,
            checkpoint_path=ckpt_path,
            device=device,
            strict=strict_load,
        )

        model_max_seq_len = load_meta.get("max_seq_len", None)
        if model_max_seq_len is None:
            model_max_seq_len = getattr(model_cfg, "max_seq_len", None)

        if model_max_seq_len is None:
            raise ValueError(
                f"Could not infer max_seq_len for {arch}. "
                "Please make sure your config has max_seq_len."
            )

        model_name = make_model_name_from_checkpoint(arch, ckpt_path)

        # Optional: shorter lengths if the model context is smaller.
        model_lengths = lengths
        if model_lengths is None:
            model_lengths = [
                128,
                256,
                512,
                min(900, int(model_max_seq_len) - 64),
            ]

        model_lengths = [
            int(x)
            for x in model_lengths
            if x > 64 and x < int(model_max_seq_len)
        ]

        if not model_lengths:
            raise ValueError(
                f"No valid probe lengths for {arch}; max_seq_len={model_max_seq_len}"
            )

        model_eff_lengths = efficiency_context_lengths
        if model_eff_lengths is None:
            model_eff_lengths = [
                128,
                256,
                512,
                768,
                min(1024, int(model_max_seq_len)),
            ]

        model_eff_lengths = sorted(set([
            int(x)
            for x in model_eff_lengths
            if x > 0 and x <= int(model_max_seq_len)
        ]))

        extra_metadata = {
            "checkpoint_path": str(ckpt_path),
            "dataset": cfg.DATASET,
            "eval_batch_size_cfg": cfg.EVAL_BATCH_SIZE,
            "eval_batches_cfg": cfg.EVAL_BATCHES,
            "val_monitor_fraction": cfg.VAL_MONITOR_FRACTION,
            "val_ablation_fraction": cfg.VAL_ABLATION_FRACTION,
            "test_replication_fraction": cfg.TEST_REPLICATION_FRACTION,
        }

        train_tokens = None
        val_loss = None
        checkpoint_step = None

        try:
            raw_ckpt = torch.load(ckpt_path, map_location="cpu")
            for key in ["train_tokens", "tokens_seen", "total_tokens"]:
                if key in raw_ckpt:
                    train_tokens = raw_ckpt[key]
                    break

            for key in ["best_val_loss", "val_loss", "loss"]:
                if key in raw_ckpt:
                    val_loss = raw_ckpt[key]
                    break

            for key in ["step", "global_step", "checkpoint_step"]:
                if key in raw_ckpt:
                    checkpoint_step = raw_ckpt[key]
                    break

            del raw_ckpt
        except Exception as e:
            print(f"Could not read optional training metadata from checkpoint: {e}")

        print(f"Model name: {model_name}")
        print(f"max_seq_len: {model_max_seq_len}")
        print(f"probe lengths: {model_lengths}")
        print(f"efficiency lengths: {model_eff_lengths}")

        result = run_full_probe_evaluation(
            model=model,
            arch=arch,
            tokenizer=tokenizer,
            device=device,
            max_seq_len=int(model_max_seq_len),
            model_name=model_name,

            out_dir=out_dir / model_name,

            config=model_cfg,
            train_tokens=train_tokens,
            val_loss=val_loss,
            checkpoint_step=checkpoint_step,
            extra_metadata=extra_metadata,

            amp_dtype=amp_dtype,
            use_amp=use_amp,
            score_by=score_by,

            passkey_seeds=passkey_seeds,
            entity_seeds=entity_seeds,
            mc_seeds=mc_seeds,

            lengths=model_lengths,
            positions=positions or [0.05, 0.25, 0.50, 0.75, 0.95],

            run_generation_examples=run_generation_examples,
            n_generation_examples=n_generation_examples,

            run_efficiency=run_efficiency,
            efficiency_context_lengths=model_eff_lengths,
        )

        all_results[arch] = result
        all_metadata.append(result["metadata"])

        all_forced_rows.extend(result.get("forced_choice_rows", []))
        all_mc_rows.extend(result.get("mc_rows", []))
        all_generation_rows.extend(result.get("generation_rows", []))
        all_efficiency_rows.extend(result.get("efficiency_rows", []))

        print(f"Finished {arch}")

        clear_model_from_memory(model=model, device=device)

    # Save combined outputs

    combined_dir = out_dir / "combined"
    combined_dir.mkdir(parents=True, exist_ok=True)

    if all_forced_rows:
        forced_df = pd.DataFrame(all_forced_rows)
        forced_path = combined_dir / "combined_forced_choice_results.csv"
        forced_df.to_csv(forced_path, index=False)
        print(f"Wrote {forced_path}")
    else:
        forced_df = pd.DataFrame()

    if all_mc_rows:
        mc_df = pd.DataFrame(all_mc_rows)
        mc_path = combined_dir / "combined_mc_results.csv"
        mc_df.to_csv(mc_path, index=False)
        print(f"Wrote {mc_path}")
    else:
        mc_df = pd.DataFrame()

    if all_generation_rows:
        generation_df = pd.DataFrame(all_generation_rows)
        generation_path = combined_dir / "combined_qualitative_generation_results.csv"
        generation_df.to_csv(generation_path, index=False)
        print(f"Wrote {generation_path}")
    else:
        generation_df = pd.DataFrame()

    if all_efficiency_rows:
        efficiency_df = pd.DataFrame(all_efficiency_rows)
        efficiency_path = combined_dir / "combined_efficiency_results.csv"
        efficiency_df.to_csv(efficiency_path, index=False)
        print(f"Wrote {efficiency_path}")
    else:
        efficiency_df = pd.DataFrame()

    metadata_path = combined_dir / "combined_metadata.json"
    metadata_path.write_text(
        json.dumps(all_metadata, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(f"Wrote {metadata_path}")

    # Save combined summary tables

    summary_tables = {}

    if len(forced_df) > 0:
        summary_forced = (
            forced_df
            .groupby(["model_name", "arch", "test_type"], dropna=False)
            .agg(
                n=("is_correct", "count"),
                accuracy=("is_correct", "mean"),
                mean_margin=("margin", "mean"),
                median_margin=("margin", "median"),
                mean_correct_score=("correct_score", "mean"),
                mean_best_wrong_score=("best_wrong_score", "mean"),
            )
            .reset_index()
            .sort_values(["test_type", "accuracy"], ascending=[True, False])
        )

        summary_tables["forced_by_test_type"] = summary_forced
        summary_forced.to_csv(combined_dir / "summary_forced_by_test_type.csv", index=False)

        passkey_df = forced_df[
            forced_df["test_type"].astype(str).str.contains("passkey", na=False)
        ]

        if len(passkey_df) > 0:
            passkey_by_length = (
                passkey_df
                .groupby(
                    ["model_name", "arch", "test_type", "target_context_tokens"],
                    dropna=False,
                )
                .agg(
                    n=("is_correct", "count"),
                    accuracy=("is_correct", "mean"),
                    mean_margin=("margin", "mean"),
                    median_margin=("margin", "median"),
                )
                .reset_index()
                .sort_values(["test_type", "target_context_tokens", "model_name"])
            )

            summary_tables["passkey_by_length"] = passkey_by_length
            passkey_by_length.to_csv(combined_dir / "summary_passkey_by_length.csv", index=False)

            passkey_by_position = (
                passkey_df[passkey_df["needle_position"].notna()]
                .groupby(
                    ["model_name", "arch", "test_type", "needle_position"],
                    dropna=False,
                )
                .agg(
                    n=("is_correct", "count"),
                    accuracy=("is_correct", "mean"),
                    mean_margin=("margin", "mean"),
                    median_margin=("margin", "median"),
                )
                .reset_index()
                .sort_values(["test_type", "needle_position", "model_name"])
            )

            summary_tables["passkey_by_position"] = passkey_by_position
            passkey_by_position.to_csv(combined_dir / "summary_passkey_by_position.csv", index=False)

    if len(mc_df) > 0:
        summary_mc = (
            mc_df
            .groupby(["model_name", "arch", "mc_scoring_mode", "test_type"], dropna=False)
            .agg(
                n=("is_correct", "count"),
                accuracy=("is_correct", "mean"),
                mean_margin=("margin", "mean"),
                median_margin=("margin", "median"),
            )
            .reset_index()
            .sort_values(
                ["mc_scoring_mode", "test_type", "accuracy"],
                ascending=[True, True, False],
            )
        )

        summary_tables["mc_by_scoring_mode"] = summary_mc
        summary_mc.to_csv(combined_dir / "summary_mc_by_scoring_mode.csv", index=False)

    if len(efficiency_df) > 0:
        summary_efficiency = (
            efficiency_df
            .groupby(["model_name", "arch", "seq_len"], dropna=False)
            .agg(
                mean_tokens_per_sec=("tokens_per_sec", "mean"),
                mean_ms_per_forward=("ms_per_forward", "mean"),
                mean_peak_mem_mb=("peak_mem_mb", "mean"),
            )
            .reset_index()
            .sort_values(["seq_len", "model_name"])
        )

        summary_tables["efficiency"] = summary_efficiency
        summary_efficiency.to_csv(combined_dir / "summary_efficiency.csv", index=False)

    print("=" * 80)
    print("All-model probe evaluation complete.")
    print(f"Combined outputs saved to: {combined_dir}")
    print("=" * 80)

    return {
        "results_by_arch": all_results,
        "combined_dir": combined_dir,
        "forced_df": forced_df,
        "mc_df": mc_df,
        "generation_df": generation_df,
        "efficiency_df": efficiency_df,
        "summary_tables": summary_tables,
        "metadata": all_metadata,
    }

In [11]:
from transformers import AutoTokenizer

TOKENIZER_NAME = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

all_results = run_all_models_probe_evaluation(
    cfg=CFG,
    tokenizer=tokenizer,
    quick_debug=False,

    passkey_seeds=list(range(20)),
    entity_seeds=list(range(100)),
    mc_seeds=list(range(100)),

    lengths=None,
    positions=[0.05, 0.25, 0.50, 0.75, 0.95],

    run_generation_examples=True,
    n_generation_examples=30,

    run_efficiency=True,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Running probe evaluation for all configured models
Device: cuda
AMP dtype: torch.bfloat16, use_amp=True
Output dir: /content/drive/MyDrive/models/eval_outputs/improved_probe_outputs

Loading transformer
Checkpoint: /content/drive/MyDrive/models/transformer_toy2/checkpoints/toy_rope_transformer_half_param_matched_tinystories_20260503_085746/best.pt
Model name: transformer_toy_rope_transformer_half_param_matched_tinystories_20260503_085746
max_seq_len: 1024
probe lengths: [128, 256, 512, 900]
efficiency lengths: [128, 256, 512, 768, 1024]
Running probe evaluation for transformer_toy_rope_transformer_half_param_matched_tinystories_20260503_085746 / transformer
Run ID: 2026-05-04_22-46-25_transformer_toy_rope_transformer_half_param_matched_tinystories_20260503_085746_transformer_28739328
Output directory: /content/drive/MyDrive/models/eval_outputs/improved_probe_outputs/transformer_toy_rope_transformer_half_param_matched_tinystories_20260503_085746
Passkey tests: 400
Passkey distractor tes